# Diagnóstico de falla en pista interna de rodamiento mediante análisis vibracional

Trabajo Práctico de Cursada — Laboratorio de Procesamiento de Señales (ITBA)

Louzao Gonzalo (63010) · Facundo Juli (63379)

---

## Resumen

Trabajamos con registros de aceleración de dos ventiladores industriales que usan rodamientos de rodillos a rótula SKF 22230 CCK/W33, medidos a $f_s = 24$ kHz por el sistema de monitoreo de la planta. Queremos saber si alguno de los puntos medidos tiene una falla localizada y en qué elemento del rodamiento, sacando la respuesta de la señal y no de una geometría que tengamos que suponer.

Para eso usamos **análisis de envolvente**. Una falla localizada no genera una sinusoide a la frecuencia de paso del defecto, sino un tren de impactos que excita resonancias de la estructura en alta frecuencia. En el espectro común esa energía queda repartida en la banda de la resonancia y no se distingue del resto, pero se recupera si se demodula esa banda. La parte difícil es decidir qué banda demodular. En vez de elegirla a ojo, la elegimos automáticamente con la kurtosis espectral (kurtograma rápido de Antoni).

Los pasos del notebook son:

1. Carga y control de calidad de los registros.
2. Caracterización del piso de ruido del sensor.
3. Indicadores temporales (RMS, cresta, kurtosis, asimetría).
4. Espectro clásico: Welch y modelo AR (Yule-Walker).
5. Elección de la banda de demodulación con el kurtograma rápido.
6. Análisis de envolvente (Hilbert) y cepstrum.
7. Identificación empírica de la falla y comparación con el reporte del fabricante.
8. Tiempo-frecuencia: STFT y wavelet de Morlet.
9. Indicadores de salud, umbral de alarma y curvas ROC y DET.
10. Tabla resumen y veredicto automático.

En cuanto a los contenidos de la materia, el trabajo usa procesos estocásticos, ruido y autocorrelación (paso 2), periodograma, ventanas y Welch (pasos 2 y 4), métodos paramétricos AR (paso 4) y análisis tiempo-frecuencia (paso 8), todo aplicado a datos industriales reales.

---

## 0. Configuración

Todos los parámetros del análisis están en esta celda.

Los registros completos duran varios minutos (en el paso 1 se imprime cuánto dura cada uno), pero por defecto analizamos 60 s (`T_ANALISIS`). Con eso el espectro de envolvente tiene una resolución de $\Delta f = f_s/N_{\text{seg}} = 24000/2^{16} = 0.366$ Hz, que alcanza de sobra para separar bandas laterales espaciadas $f_r \approx 16.7$ Hz. En el paso 7 verificamos que el resultado no depende de este valor.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal, stats, linalg

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---------------------------------------------------------------- rutas y formato
DATA_DIR     = Path("datos")      # carpeta con los CSV, relativa al notebook
CSV_SEP      = ";"                # separador de campos
CSV_DECIMAL  = ","                # separador decimal
CSV_ENCODING = "utf-8"            # si falla, el loader reintenta con latin-1

# ---------------------------------------------------------------- adquisicion
FS_ACEL = 24000.0    # Hz, forma de onda de aceleracion
FS_LF   = 3000.0     # Hz, formas de onda de velocidad y envolvente (ya procesadas)
T_ANALISIS = 60.0    # s de señal a procesar

# ---------------------------------------------------------------- maquina
RPM_NOMINAL = 1000.0            # rpm del ventilador con falla durante la medicion
FR_NOMINAL  = RPM_NOMINAL / 60  # Hz, frecuencia de giro del eje

# ---------------------------------------------------------------- analisis
NPERSEG_PSD = 2 ** 14   # ventana de Welch para la PSD de la señal cruda
NPERSEG_ENV = 2 ** 16   # ventana de Welch para el espectro de envolvente
T_VENTANA   = 1.0       # s, ventana para los indicadores por segmento
K_SIGMA     = 3.0       # margen del umbral de alarma, en sigmas de la referencia

# ---------------------------------------------------------------- graficos
plt.rcParams.update({
    "figure.figsize": (11, 4),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
})

COLOR = {"falla": "#c0392b", "sano": "#2471a3", "ref": "#27ae60",
         "ref2": "#16a085", "ruido": "#7f8c8d"}

import scipy
print(f"NumPy {np.__version__} | SciPy {scipy.__version__} | pandas {pd.__version__}")
print(f"Tramo de analisis: {T_ANALISIS:.0f} s -> {int(FS_ACEL*T_ANALISIS):,} muestras/canal")
print(f"Resolucion del espectro de envolvente: {FS_ACEL/NPERSEG_ENV:.3f} Hz")

### Los registros

| Archivo | Contenido | $f_s$ | Para qué se usa |
|---|---|---|---|
| `med2_acel_wf.csv` | aceleración, 2 canales | 24 kHz | caso de estudio. Canal 1 = lado acople (con falla), canal 2 = lado libre del mismo ventilador |
| `med1_acel_wf.csv` | aceleración, 2 canales | 24 kHz | otro ventilador, aparentemente sano: referencia externa |
| `ruido_sensor_acel.csv` | aceleración, 1 canal | 24 kHz | sensor apoyado sobre un escritorio, sin máquina: piso de referencia |
| `med*_vel_wf.csv` | velocidad [mm/s] | 3 kHz | medir $f_r$ (pico $1\times$) |
| `med*_env_wf.csv` | envolvente [gE] | 3 kHz | envolvente del software de mantenimiento, para la validación cruzada del paso 6 |

Las señales de velocidad y envolvente ya vienen procesadas por el software de mantenimiento predictivo, así que no las usamos como entrada del análisis. La velocidad solo sirve para medir $f_r$ con precisión, y la envolvente del software la usamos al final del paso 6 para comprobar que nuestra demodulación llega al mismo resultado.

### Rodamiento y frecuencias características

Es un SKF 22230 CCK/W33 (rodillos a rótula, doble hilera; según el reporte SKF, $d=150$ mm, $D=270$ mm, $B=73$ mm). Las frecuencias de falla dependen de la geometría interna:

$$
\begin{aligned}
\text{BPFO} &= \tfrac{Z}{2}\,f_r\left(1 - \tfrac{D_w}{D_{pw}}\cos\alpha\right)
   &&\text{(paso por pista externa)}\\[2pt]
\text{BPFI} &= \tfrac{Z}{2}\,f_r\left(1 + \tfrac{D_w}{D_{pw}}\cos\alpha\right)
   &&\text{(paso por pista interna)}\\[2pt]
\text{BSF}  &= \tfrac{D_{pw}}{2D_w}\,f_r\left(1 - \tfrac{D_w^2}{D_{pw}^2}\cos^2\alpha\right)
   &&\text{(giro del rodillo sobre su eje)}\\[2pt]
\text{FTF}  &= \tfrac{1}{2}\,f_r\left(1 - \tfrac{D_w}{D_{pw}}\cos\alpha\right)
   &&\text{(jaula)}
\end{aligned}
$$

donde $Z$ es el número de elementos rodantes, $D_w$ el diámetro del rodillo, $D_{pw}$ el diámetro primitivo y $\alpha$ el ángulo de contacto.

El problema es que no conocemos $Z$, $D_w$, $D_{pw}$ ni $\alpha$, porque SKF no publica la geometría interna. Lo que sí tenemos es el reporte *SKF Product select* (`SKF_Bearing_calculation_report-1_260916_125317.pdf`), que para $n = 995$ rpm da directamente las frecuencias calculadas por el fabricante. De ahí despejamos los **coeficientes adimensionales** $f/f_r$, que son los que importan porque no dependen de la velocidad de giro. La celda siguiente los calcula y verifica dos identidades de consistencia:

$$\text{BPFO} + \text{BPFI} = Z\,f_r \qquad\text{y}\qquad \text{BPFO} = Z\cdot\text{FTF}$$

que además sirven para recuperar $Z$ sin conocer la geometría.

In [ ]:
# --- datos transcritos del reporte SKF Product select (22230 CCK/C3 W33, n = 995 rpm) ---
SKF = {
    "n_rpm": 995.0,
    "f_i":  16.583,   # giro del aro interno = fr
    "f_c":   7.135,   # jaula (FTF)
    "f_r":  57.425,   # giro del rodillo sobre su propio eje (BSF)
    "f_ip": 179.524,  # sobre-rodadura de un punto de la pista interna (BPFI)
    "f_ep": 135.559,  # sobre-rodadura de un punto de la pista externa (BPFO)
    "f_rp": 114.850,  # sobre-rodadura del elemento rodante (= 2*BSF)
}

fr_skf = SKF["f_i"]

# coeficientes adimensionales: multiplos de la frecuencia de giro
COEF = {
    "FTF":   SKF["f_c"]  / fr_skf,
    "BSF":   SKF["f_r"]  / fr_skf,
    "2xBSF": SKF["f_rp"] / fr_skf,
    "BPFO":  SKF["f_ep"] / fr_skf,
    "BPFI":  SKF["f_ip"] / fr_skf,
}

# --- consistencia: de ellas se deduce Z sin conocer la geometria interna ---
Z_de_suma = (SKF["f_ip"] + SKF["f_ep"]) / fr_skf     # BPFO + BPFI = Z*fr
Z_de_ftf  = SKF["f_ep"] / SKF["f_c"]                 # BPFO = Z*FTF
Z = int(round(Z_de_suma))

print("Coeficientes del rodamiento SKF 22230 CCK/W33 (multiplos de fr)")
print("-" * 58)
for k, v in COEF.items():
    print(f"  {k:<6} = {v:7.4f} x fr")
print("-" * 58)
print(f"  Z deducido de  BPFO+BPFI = Z*fr : {Z_de_suma:.4f}")
print(f"  Z deducido de  BPFO      = Z*FTF: {Z_de_ftf:.4f}")
print(f"  -> Z = {Z} elementos rodantes por hilera "
      f"(discrepancia {abs(Z_de_suma - Z_de_ftf):.2e})")

gamma = 1 - 2 * COEF["FTF"]   # despejado de FTF = (fr/2)(1 - Dw/Dpw cos a)
print(f"  (Dw/Dpw)*cos(alpha) = {gamma:.4f}")
print(f"  BPFI / (Z*fr) = {COEF['BPFI'] / Z:.3f}   (regla practica: BPFI ~ 0.6*Z*fr)")
print(f"\nFrecuencias calculadas por SKF a n = {SKF['n_rpm']:.0f} rpm "
      f"(fr = {fr_skf:.3f} Hz)")

Una aclaración sobre el rango de BPFI. Es común ver publicado que BPFI está entre $5$ y $8\times f_r$, pero ese rango es para rodamientos rígidos de bolas con $Z \approx 8$–$12$ elementos: la regla práctica $\text{BPFI} \approx 0.6\,Z\,f_r$ da $4.8$–$7.2\times f_r$. Este rodamiento tiene $Z = 19$ rodillos por hilera y un coeficiente de $0.570\,Z\,f_r$ (impreso arriba), así que su BPFI queda en $\mathbf{10.83 \times f_r}$. Si usáramos el rango genérico para decidir, descartaríamos el pico correcto. Por eso en el paso 7 buscamos hasta $15\times f_r$ y comparamos contra los coeficientes del reporte, no contra un rango de manual.

---

## Funciones base

Definimos las funciones una sola vez acá, así cada paso queda corto y, sobre todo, todos los canales se procesan con **exactamente los mismos parámetros**. Si no fuera así, comparar falla, sano y ruido no tendría sentido.

In [ ]:
def cargar_wf(nombre, fs_esperada, t_max=None):
    # Lee una forma de onda exportada por el sistema de monitoreo.
    # Devuelve (t, X, fs, etiquetas) con X de forma (n_muestras, n_canales).
    # Las columnas se acceden por POSICION: la 0 es el tiempo, el resto son canales.
    ruta = Path(DATA_DIR) / nombre
    nrows = None if t_max is None else int(fs_esperada * t_max)
    df = None
    for enc in (CSV_ENCODING, "latin-1"):
        try:
            df = pd.read_csv(ruta, sep=CSV_SEP, decimal=CSV_DECIMAL,
                             encoding=enc, nrows=nrows)
            break
        except UnicodeDecodeError:
            continue
    if df is None:
        raise IOError(f"no se pudo decodificar {ruta}")

    t = df.iloc[:, 0].to_numpy(float)
    X = df.iloc[:, 1:].to_numpy(float)

    # Verificacion de fs contra el propio vector de tiempo. Se usa una regresion lineal
    # sobre TODO el vector y no la mediana de diff(t): la columna Tiempo esta exportada
    # con ~7 cifras significativas, de modo que cada dt individual esta cuantizado
    # (4.17e-5 en vez de 4.1666...e-5) y su mediana da 23980.8 Hz en vez de 24000 Hz.
    fs_medida = 1.0 / np.polyfit(np.arange(len(t)), t, 1)[0]
    if abs(fs_medida - fs_esperada) / fs_esperada > 1e-3:
        print(f"  AVISO {nombre}: fs medida {fs_medida:.1f} Hz != "
              f"esperada {fs_esperada:.1f} Hz")
    etiquetas = [str(c).split("\\")[0].strip() for c in df.columns[1:]]
    return t, X, fs_medida, etiquetas


def preparar(x):
    # Quita offset de continua y tendencia lineal. Imprescindible antes de cualquier
    # estadistico de forma: un offset infla el RMS, reduce artificialmente el factor
    # de cresta y sesga la kurtosis.
    return signal.detrend(np.asarray(x, float), type="linear")


def segmentar(x, fs, t_seg=T_VENTANA, solape=0.0):
    # Parte la señal en ventanas de t_seg segundos -> (n_ventanas, n_muestras).
    n = int(fs * t_seg)
    paso = max(1, int(n * (1 - solape)))
    return np.array([x[i:i + n] for i in range(0, len(x) - n + 1, paso)])


def indicadores_tiempo(x):
    x = np.asarray(x, float)
    rms = np.sqrt(np.mean(x ** 2))
    pico = np.max(np.abs(x))
    return {
        "RMS": rms,
        "Pico": pico,
        "Cresta": pico / rms if rms > 0 else np.nan,
        "Kurtosis": stats.kurtosis(x, fisher=True, bias=False),
        "Skewness": stats.skew(x, bias=False),
    }


def psd_welch(x, fs, nperseg=NPERSEG_PSD, average="mean", ventana="hann"):
    # PSD unilateral por el metodo de Welch, con solape del 50 %.
    return signal.welch(x, fs=fs, window=ventana, nperseg=nperseg,
                        noverlap=nperseg // 2, average=average)


def espectro_envolvente(x, fs, banda, orden=6, nperseg=NPERSEG_ENV):
    # Demodulacion de amplitud: pasabanda -> señal analitica (Hilbert) -> |.| -> PSD.
    lo = max(float(banda[0]), 1.0)
    hi = min(float(banda[1]), 0.99 * fs / 2)
    sos = signal.butter(orden, [lo, hi], btype="bandpass", fs=fs, output="sos")
    y = signal.sosfiltfilt(sos, x)
    env = np.abs(signal.hilbert(y))
    env = env - env.mean()          # quita la continua de la envolvente
    f, P = signal.welch(env, fs=fs, nperseg=nperseg, noverlap=nperseg // 2)
    return f, P, env, y


def pico_cerca(f, P, f_objetivo, tol):
    # (frecuencia, amplitud, relacion pico/fondo en dB) del maximo en f_objetivo +- tol.
    # El "fondo" es la mediana de la PSD en un entorno 10 veces mas ancho.
    m = np.abs(f - f_objetivo) <= tol
    if not m.any():
        return np.nan, np.nan, np.nan
    j = np.argmax(P[m])
    f_pico, a_pico = f[m][j], P[m][j]
    fondo = np.median(P[np.abs(f - f_objetivo) <= 10 * tol])
    snr = 10 * np.log10(a_pico / fondo) if fondo > 0 else np.nan
    return f_pico, a_pico, snr


def decimar_minmax(x, n_out=2000):
    # Reduce una señal larga a ~n_out pares (min, max) por bloque, para graficarla sin
    # dibujar millones de puntos y sin perder la envolvente visual de la forma de onda.
    n = len(x)
    bloque = max(1, n // n_out)
    m = (n // bloque) * bloque
    B = np.asarray(x[:m], float).reshape(-1, bloque)
    idx = (np.arange(B.shape[0]) + 0.5) * bloque
    return idx, B.min(axis=1), B.max(axis=1)


def marcar(ax, freqs, etiquetas=None, color="k", ls="--", alpha=0.6, y_rel=0.97):
    # Lineas verticales anotadas (fr, armonicos, BPFI, ...).
    etiquetas = etiquetas if etiquetas is not None else [None] * len(freqs)
    for fq, et in zip(freqs, etiquetas):
        ax.axvline(fq, color=color, ls=ls, lw=0.9, alpha=alpha)
        if et:
            ax.annotate(et, xy=(fq, y_rel), xycoords=("data", "axes fraction"),
                        rotation=90, va="top", ha="right", fontsize=7.5,
                        color=color, alpha=0.95)


print("funciones base definidas")

---

# 1. Carga y control de calidad

Antes de calcular cualquier indicador hay que revisar tres cosas de los datos.

La primera es si la frecuencia de muestreo es la que creemos. La verificamos con el vector de tiempo del propio archivo y no con lo que dice la documentación. Hay un detalle: la columna `Tiempo` está exportada con resolución finita, así que cada intervalo queda cuantizado. La mediana de $\Delta t$ da $4.17\times10^{-5}$ s en lugar de $4.1\overline{6}\times10^{-5}$ s, y $1/\text{mediana}(\Delta t)$ da 23980.8 Hz, un error de −0.08 % que se arrastraría a todas las frecuencias del análisis (print de abajo). Con una regresión lineal sobre todo el vector se recuperan los 24000 Hz.

La segunda es si hubo **saturación** (*clipping*). Si el acelerómetro o el conversor A/D saturaron, los picos quedan recortados, y eso destruye justo lo que buscamos, la impulsividad, además de generar armónicos espurios. Con un registro saturado no sirven ni la kurtosis, ni el factor de cresta, ni la envolvente. Se detecta buscando si se acumulan muestras en el valor extremo.

La tercera es si hay offset o deriva. Un piezoeléctrico con acoplamiento AC no debería tener continua, pero la integración numérica del equipo o una deriva térmica la pueden meter. La sacamos con

$$x_{\text{prep}}[n] = x[n] - (\hat{a}n + \hat{b})$$

donde $\hat a,\hat b$ salen de un ajuste por mínimos cuadrados (`scipy.signal.detrend`). Esto importa porque un offset $\mu$ sube el RMS a $\sqrt{\mu^2+\sigma^2}$ y baja el factor de cresta, lo que puede tapar la falla.

Además partimos todas las señales en ventanas de 1 s. Así, en el paso 9, cada indicador se puede tratar como una variable aleatoria con media y dispersión, y no como un único número sin incertidumbre.

In [ ]:
# Cada canal se identifica con una clave corta que se usa en todo el notebook.
ESPEC = [
    ("M2-C1", "med2_acel_wf.csv", 0, "Vent. 2, lado acople (sospecha de falla)", "falla"),
    ("M2-C2", "med2_acel_wf.csv", 1, "Vent. 2, lado libre (mismo equipo)",       "sano"),
    ("M1-C1", "med1_acel_wf.csv", 0, "Vent. 1, punto A (equipo de referencia)",  "ref"),
    ("M1-C2", "med1_acel_wf.csv", 1, "Vent. 1, punto B (equipo de referencia)",  "ref2"),
    ("RUIDO", "ruido_sensor_acel.csv", 0, "Sensor sobre escritorio, sin maquina",  "ruido"),
]

CH = {}
_cache = {}
for clave, archivo, icol, descripcion, rol in ESPEC:
    if archivo not in _cache:
        _cache[archivo] = cargar_wf(archivo, FS_ACEL, t_max=T_ANALISIS)
    t, X, fs, etiquetas = _cache[archivo]
    bruto = X[:, icol]
    CH[clave] = {
        "t": t, "bruto": bruto, "x": preparar(bruto), "fs": fs,
        "archivo": archivo, "desc": descripcion, "rol": rol,
        "etiqueta": etiquetas[icol] if icol < len(etiquetas) else f"canal {icol+1}",
    }
del _cache

filas = []
for clave, d in CH.items():
    x, b = d["x"], d["bruto"]
    vmax = np.max(np.abs(b))
    # proxy de clipping: fraccion de muestras a menos de 0.1 % del extremo
    frac_extremo = np.mean(np.abs(b) >= 0.999 * vmax)
    filas.append({
        "canal": clave,
        "archivo": d["archivo"],
        "fs [Hz]": round(d["fs"], 1),
        "N": len(x),
        "T [s]": round(len(x) / d["fs"], 2),
        "DC [g]": f"{b.mean():+.2e}",
        "min [g]": round(b.min(), 3),
        "max [g]": round(b.max(), 3),
        "frac. en extremo": f"{frac_extremo:.2e}",
    })

qc = pd.DataFrame(filas).set_index("canal")
display(qc)

n_extremo = {k: int(np.sum(np.abs(d["bruto"]) >= 0.999 * np.max(np.abs(d["bruto"]))))
             for k, d in CH.items()}
print("Muestras en el valor extremo (esperado 1-2 si NO hay saturacion):")
print("  " + ",  ".join(f"{k}: {v}" for k, v in n_extremo.items()))

# fs: la mediana de diff(t) esta cuantizada por la resolucion con que se exporto Tiempo
t_ref = CH["M2-C1"]["t"]
dt_med = np.median(np.diff(t_ref))
print(f"\nPaso de tiempo mediano: {dt_med:.4e} s   (1/{FS_ACEL:.0f} = {1 / FS_ACEL:.4e} s)")
print(f"fs por 1/mediana(dt): {1 / dt_med:.1f} Hz   fs por regresion: "
      f"{CH['M2-C1']['fs']:.1f} Hz   -> error de la mediana "
      f"{100 * (1 / dt_med - FS_ACEL) / FS_ACEL:+.2f} %")

# duracion de los registros completos: alcanza con leer la ultima linea de cada archivo
print(f"\nDuracion de los registros completos (se analizan los primeros {T_ANALISIS:.0f} s):")
for archivo in sorted({d["archivo"] for d in CH.values()}):
    with open(Path(DATA_DIR) / archivo, "rb") as fh:
        fh.seek(-200, 2)
        ultima = fh.read().decode("latin-1").strip().splitlines()[-1]
    t_fin = float(ultima.split(CSV_SEP)[0].replace(CSV_DECIMAL, "."))
    print(f"  {archivo:<24} {t_fin:6.1f} s  ->  {round(t_fin * FS_ACEL) + 1:,} muestras/canal")

In [ ]:
fig, axes = plt.subplots(len(CH), 2, figsize=(12.5, 2.0 * len(CH)),
                         gridspec_kw={"width_ratios": [2, 1]})
for ax_row, (clave, d) in zip(axes, CH.items()):
    x, fs = d["x"], d["fs"]
    c = COLOR[d["rol"]]

    # --- izquierda: 0.5 s de señal temporal ---
    n = int(0.5 * fs)
    ax_row[0].plot(np.arange(n) / fs * 1e3, x[:n], color=c, lw=0.6)
    ax_row[0].set_ylabel(f"{clave}\n[g]")
    ax_row[0].set_xlim(0, 500)
    if clave == list(CH)[0]:
        ax_row[0].set_title("Señal temporal (primeros 500 ms)")

    # --- derecha: histograma en escala log, para ver colas y saturacion ---
    ax_row[1].hist(x, bins=200, color=c, log=True, alpha=0.85)
    ax_row[1].set_ylabel("cuentas")
    if clave == list(CH)[0]:
        ax_row[1].set_title("Histograma de amplitudes (eje y logaritmico)")

axes[-1][0].set_xlabel("Tiempo [ms]")
axes[-1][1].set_xlabel("Aceleracion [g]")
fig.tight_layout()
plt.show()

*Figura 1. A la izquierda, los primeros 500 ms de cada canal, sin continua ni tendencia. A la derecha, el histograma de amplitudes con eje vertical logarítmico para que se vean las colas. Ningún histograma acumula cuentas en los extremos (un registro saturado tendría dos barras aisladas en $\pm V_{\max}$) y cada canal tiene una sola muestra en su valor extremo, así que no hay clipping. Las escalas cambian entre filas: los canales de máquina tienen picos de 11 a 19 g y el ruido, de 0.046 g.*

In [ ]:
# Vista del registro COMPLETO, con decimacion min/max para no dibujar 1.4 M de puntos.
# Sirve para juzgar estacionariedad: si el nivel cambia a lo largo del registro, promediar
# sobre todo el tramo (Welch, kurtosis global) deja de tener sentido.
fig, axes = plt.subplots(len(CH), 1, figsize=(12, 1.55 * len(CH)), sharex=True)
for a, (clave, d) in zip(axes, CH.items()):
    idx, lo, hi = decimar_minmax(d["x"], 2500)
    t_s = idx / d["fs"]
    a.fill_between(t_s, lo, hi, color=COLOR[d["rol"]], lw=0)
    a.set_ylabel(f"{clave}\n[g]")
    a.margins(x=0)
axes[0].set_title(f"Registro completo ({T_ANALISIS:.0f} s) — envolvente min/max de la "
                  f"forma de onda")
axes[-1].set_xlabel("Tiempo [s]")
fig.tight_layout()
plt.show()

print("Estacionariedad: RMS por ventana de 1 s")
rms_glob = {}
for clave, d in CH.items():
    rv = np.sqrt(np.mean(segmentar(d["x"], d["fs"], 1.0) ** 2, axis=1))
    rms_glob[clave] = np.sqrt(np.mean(d["x"] ** 2))
    print(f"  {clave:<6}: {rv.mean():.4f} +- {rv.std():.4f} g "
          f"(dispersion {100*rv.std()/rv.mean():4.1f} %)   "
          f"min {rv.min():.4f} g en t = {rv.argmin()} s, max {rv.max():.4f} g en t = {rv.argmax()} s")
razones = [rms_glob[k] / rms_glob["RUIDO"] for k in CH if k != "RUIDO"]
print(f"\nRMS de los canales de maquina / RMS del ruido: entre {min(razones):.0f} y "
      f"{max(razones):.0f} veces")

*Figura 2. Los 60 s analizados de cada canal, como envolvente mín/máx de la forma de onda. M2-C1, M2-C2 y M1-C1 mantienen el nivel (dispersión del RMS por ventana entre 1.8 y 2.7 %). M1-C2 cae al final (ventana de menor RMS en t = 58 s, con 0.43 g contra una media de 1.36 g) y RUIDO tiene su transitorio dominante cerca de t = 28 s.*

Los histogramas son campanas sin acumulación en los extremos, así que **no hay saturación**. Un registro recortado mostraría dos barras altas aisladas en $\pm V_{\max}$ y un corte brusco de la distribución. En cada canal hay una sola muestra exactamente en el valor extremo, que es lo esperable para un máximo estadístico y no para un recorte.

En el registro completo, M2-C1, M2-C2 y M1-C1 mantienen el nivel durante los 60 s: la dispersión del RMS entre ventanas está entre 1.8 y 2.7 %. Por eso los podemos tratar como aproximadamente estacionarios y promediar sobre todo el tramo, que es lo que hacen Welch y los indicadores globales.

Hay dos excepciones. RUIDO tiene su ventana de mayor RMS en t = 28 s (dispersión del RMS de 28.0 %); es el *glitch* que caracterizamos en el paso 2 y la razón del promediado robusto. M1-C2 baja de nivel al final del registro: su ventana de menor RMS es la de t = 58 s, con 0.43 g contra una media de 1.36 g (dispersión de 12.6 %, frente a 1.8–2.7 % de los demás). Puede ser el final de la adquisición o un cambio en la condición de la máquina. No afecta el diagnóstico, porque M1-C2 es solo un canal de referencia, pero explica los valores atípicos que aparecen después en el RMS por ventana.

El canal de ruido tiene un RMS entre 581 y 1465 veces menor que los de máquina, y su histograma tiene colas muy largas para su desviación estándar; eso también es el *glitch* del paso 2.

### Estimación de la frecuencia de giro $f_r$

El equipo indica 1000 rpm, o sea $f_r = 16.667$ Hz. Preferimos **medirla** en lugar de suponerla, porque todo el diagnóstico se expresa en múltiplos de $f_r$ y un error del 1 % en $f_r$ es un error del 1 % en la frecuencia de falla.

Para esto sirve la señal de velocidad. Integrar ($v = \int a\,dt$) es como aplicar un filtro $1/f$ que realza las bajas frecuencias, así que el pico $1\times$ (el desbalanceo residual que tiene cualquier máquina rotativa) domina el espectro.

In [ ]:
FR = {}
for arch, equipo in [("med2_vel_wf.csv", "Vent. 2"), ("med1_vel_wf.csv", "Vent. 1")]:
    _, V, fsv, _ = cargar_wf(arch, FS_LF, t_max=min(T_ANALISIS * 2, 120.0))
    picos = []
    for i in range(V.shape[1]):
        f, P = psd_welch(preparar(V[:, i]), fsv, nperseg=2 ** 15)
        m = (f > 0.6 * FR_NOMINAL) & (f < 1.6 * FR_NOMINAL)   # entorno de 1x
        picos.append(f[m][np.argmax(P[m])])
    FR[arch[:4]] = float(np.mean(picos))
    print(f"{equipo} ({arch}): 1x = {np.mean(picos):.3f} Hz "
          f"-> {np.mean(picos)*60:.1f} rpm   (canales: "
          f"{', '.join(f'{p:.3f}' for p in picos)} Hz)")

fr = FR["med2"]           # frecuencia de giro del equipo bajo estudio
print(f"\nSe adopta fr = {fr:.3f} Hz "
      f"(desvio respecto del nominal {FR_NOMINAL:.3f} Hz: "
      f"{100*(fr-FR_NOMINAL)/FR_NOMINAL:+.2f} %)")

# frecuencias de falla teoricas escaladas a la fr medida
TEO = {k: v * fr for k, v in COEF.items()}
print("\nFrecuencias de falla teoricas a fr medida:")
for k, v in TEO.items():
    print(f"  {k:<6} = {v:8.2f} Hz  ({COEF[k]:6.3f} x fr)")

---

# 2. Caracterización del ruido del sensor

La medición de ruido (el sensor apoyado sobre un escritorio, sin máquina) es el **piso** contra el que medimos todo lo demás. Sin ella no podemos decir que una componente espectral existe, solo que está por encima de algo.

### PSD por el método de Welch

El periodograma $\hat{P}(f) = |X(f)|^2/(f_s N)$ es un estimador inconsistente: su varianza no baja aunque aumente $N$. Welch la reduce partiendo la señal en $K$ segmentos solapados, ventaneando cada uno y promediando:

$$\hat{P}_W(f) \;=\; \frac{1}{K}\sum_{k=1}^{K}
\frac{\left|\sum_{n=0}^{L-1} w[n]\,x_k[n]\,e^{-j2\pi fn/f_s}\right|^2}{f_s\sum_{n} w^2[n]}$$

A cambio se pierde resolución, $\Delta f = f_s/L$. Con $L = 2^{14}$ queda $\Delta f = 1.46$ Hz, suficiente para un piso de ruido, que por definición es suave.

### Por qué promediar por mediana

En el registro de ruido hay un transitorio dominante (*glitch*) de origen desconocido: puede ser un golpe en el escritorio o en el cable, un arranque cercano o una descarga. Tiene energía en un rango amplio de frecuencias y contamina los segmentos donde cae.

La media no es robusta: un segmento con energía $M$ veces mayor mueve el promedio en $M/K$. La mediana, en cambio, tiene un punto de ruptura del 50 %, o sea que hay que contaminar la mitad de los segmentos para moverla. En SciPy es `welch(..., average='median')`. Calculamos las dos para ver cuánto cambia en estos datos.

In [ ]:
rui = CH["RUIDO"]["x"]
fs = CH["RUIDO"]["fs"]

# --- localizacion del glitch: RMS por ventana corta ---
V = segmentar(rui, fs, t_seg=0.1)
rms_v = np.sqrt(np.mean(V ** 2, axis=1))
t_v = np.arange(len(rms_v)) * 0.1
i_glitch = int(np.argmax(rms_v))

print(f"RMS global del ruido        : {np.sqrt(np.mean(rui**2)):.3e} g")
print(f"RMS mediano por ventana     : {np.median(rms_v):.3e} g")
print(f"RMS de la ventana mas ruidosa: {rms_v[i_glitch]:.3e} g "
      f"(t = {t_v[i_glitch]:.1f} s, {rms_v[i_glitch]/np.median(rms_v):.1f}x la mediana)")
print(f"Kurtosis del registro        : {stats.kurtosis(rui, bias=False):.2f}  "
      f"(0 para ruido gaussiano)")
for umb in (2, 3):
    t_alto = t_v[rms_v > umb * np.median(rms_v)]
    print(f"Ventanas de 100 ms con RMS > {umb}x la mediana: {len(t_alto)}  "
          f"(t = {', '.join(f'{x:.1f}' for x in t_alto)} s)")

fig, ax = plt.subplots(1, 2, figsize=(12.5, 3.2))
ax[0].plot(np.arange(len(rui)) / fs, rui, color=COLOR["ruido"], lw=0.4)
ax[0].axvline(t_v[i_glitch], color="r", ls="--", lw=1)
ax[0].annotate("glitch", xy=(t_v[i_glitch], 0.9), xycoords=("data", "axes fraction"),
               color="r", fontsize=9, ha="left")
ax[0].set(xlabel="Tiempo [s]", ylabel="Aceleracion [g]",
          title="Registro de ruido completo")
n0 = int((t_v[i_glitch]) * fs)
n1 = int((t_v[i_glitch] + 0.1) * fs)
ax[1].plot(np.arange(n0, n1) / fs, rui[n0:n1], color="r", lw=0.7)
ax[1].set(xlabel="Tiempo [s]", ylabel="Aceleracion [g]",
          title="Detalle del transitorio")
fig.tight_layout()
plt.show()

*Figura 3. Registro de ruido completo (izquierda) y los 100 ms de mayor RMS (derecha), en t = 28.0 s, con 5.3 veces el RMS mediano. Es el transitorio dominante: las cinco ventanas que superan 3 veces la mediana caen entre 27.8 y 28.5 s. Con el criterio de 2 veces aparece además una ventana suelta en t = 5.8 s.*

In [ ]:
f_n, P_media   = psd_welch(rui, fs, nperseg=NPERSEG_PSD, average="mean")
f_n, P_mediana = psd_welch(rui, fs, nperseg=NPERSEG_PSD, average="median")

# PISO DE RUIDO DE REFERENCIA: se usa en los pasos 4, 5, 9 y 10
PSD_RUIDO = P_mediana
F_RUIDO = f_n

fig, ax = plt.subplots(1, 2, figsize=(12.5, 3.8))
ax[0].loglog(f_n[1:], P_media[1:],   lw=0.8, color="#d35400", label="promedio por media")
ax[0].loglog(f_n[1:], P_mediana[1:], lw=0.8, color=COLOR["ruido"],
             label="promedio por mediana")
ax[0].set(xlabel="Frecuencia [Hz]", ylabel=r"PSD [$g^2$/Hz]",
          title="Piso de ruido del sensor (Welch)")
ax[0].legend()

razon_db = 10 * np.log10(P_media / P_mediana)
ax[1].semilogx(f_n[1:], razon_db[1:], lw=0.8, color="#d35400")
ax[1].axhline(0, color="k", lw=0.8)
ax[1].set(xlabel="Frecuencia [Hz]", ylabel="media / mediana [dB]",
          title="Sesgo que el glitch introduce en el promedio por media")
fig.tight_layout()
plt.show()

K = len(rui) // (NPERSEG_PSD // 2) - 1
print(f"Segmentos promediados: K = {K}   (L = {NPERSEG_PSD}, Delta f = {fs / NPERSEG_PSD:.2f} Hz)")
print(f"Sesgo MEDIANO en frecuencia : {np.median(razon_db):+.2f} dB")
print(f"Sesgo MAXIMO  en frecuencia : {np.max(razon_db):+.2f} dB "
      f"(en {f_n[np.argmax(razon_db)]:.0f} Hz)")
print(f"Fraccion del eje con sesgo > 3 dB: {100*np.mean(razon_db > 3):.1f} %")
c0 = int(t_v[i_glitch] * fs)                      # el mismo cociente, en 5 s alrededor del glitch
seg5 = rui[c0 - int(2.5 * fs): c0 + int(2.5 * fs)]
_, P5m = psd_welch(seg5, fs, nperseg=NPERSEG_PSD, average="mean")
_, P5d = psd_welch(seg5, fs, nperseg=NPERSEG_PSD, average="median")
print(f"Sobre 5 s centrados en el glitch: sesgo maximo {10*np.log10(np.max(P5m / P5d)):+.2f} dB")

print("\nPiso de ruido (mediana) por banda:")
for fq in [100, 500, 1000, 2000, 4000, 8000, 11000]:
    print(f"  {fq:>6} Hz : {10*np.log10(np.interp(fq, f_n, P_mediana)):7.1f} dB "
          f"re 1 g^2/Hz")
caida = (10 * np.log10(np.interp(11000, f_n, P_mediana))
         - 10 * np.log10(np.interp(100, f_n, P_mediana)))
print(f"\nCaida de 100 Hz a 11 kHz: {caida:+.1f} dB  ->  el ruido NO es blanco")

# lineas de banda angosta que sobreviven al promedio por mediana
m_nb = (f_n > 20) & (f_n < 1000)
pk_nb, pr_nb = signal.find_peaks(10 * np.log10(P_mediana[m_nb]), prominence=3)
print("\nLineas de banda angosta en el piso (prominencia > 3 dB, 20-1000 Hz):")
print("  " + ", ".join(f"{f_n[m_nb][i]:.1f} Hz ({p:.1f} dB)"
                       for i, p in zip(pk_nb, pr_nb["prominences"])))

*Figura 4. Izquierda: PSD de Welch del ruido ($L = 2^{14}$, $\Delta f$ = 1.46 Hz) promediando por media y por mediana. Derecha: el cociente entre las dos, en dB. El sesgo de la media no es parejo: su mediana es +0.03 dB, pero llega a +9.87 dB en 469 Hz, donde se concentra el transitorio, y pasa los 3 dB en el 2.0 % del eje. La curva por mediana cae 21.6 dB entre 100 Hz y 11 kHz (el ruido no es blanco) y conserva líneas angostas, como las de 82.0 y 99.6 Hz, que no vienen del glitch.*

Con 60 s de registro hay $K = 174$ segmentos y un solo transitorio dominante, así que el sesgo mediano sobre el eje de frecuencias es de apenas $+0.03$ dB. Promediando por media, el piso global casi no se mueve. El tema es que el sesgo no está repartido parejo: donde el transitorio concentra su energía llega a $+9.87$ dB (469 Hz). O sea, la media no corre el piso, pero le agrega **falsos relieves locales**, que después se podrían confundir con componentes reales. Con registros más cortos es peor: en 5 s centrados en el *glitch*, el sesgo máximo llega a $+15.47$ dB.

Como la mediana cuesta lo mismo y no tiene contras, la usamos para el piso de referencia.

### ¿Es ruido blanco?

Un proceso blanco tiene PSD constante y autocorrelación impulsiva:

$$r_{xx}[m] = \sigma^2\,\delta[m]
\quad\Longleftrightarrow\quad
P_{xx}(f) = \sigma^2 \;\; \text{constante}$$

Para decidirlo usamos la autocorrelación normalizada $\rho[m]=r_{xx}[m]/r_{xx}[0]$ y la comparamos con las bandas de confianza del 95 % para la hipótesis de blancura, $\pm 1.96/\sqrt{N}$.

In [ ]:
def autocorr(x, maxlag):
    # Autocorrelacion insesgada-en-media via FFT (Wiener-Khinchin).
    x = np.asarray(x, float) - np.mean(x)
    N = len(x)
    nfft = 1 << int(np.ceil(np.log2(2 * N)))
    S = np.abs(np.fft.rfft(x, nfft)) ** 2
    r = np.fft.irfft(S, nfft)[:maxlag + 1]
    return r / N


maxlag = 400
r_rui = autocorr(rui, maxlag)
rho = r_rui / r_rui[0]
lim = 1.96 / np.sqrt(len(rui))

fig, ax = plt.subplots(1, 2, figsize=(12.5, 3.4))
lags_ms = np.arange(maxlag + 1) / fs * 1e3
ax[0].stem(lags_ms[:60], rho[:60], basefmt=" ", markerfmt=".",
           linefmt=COLOR["ruido"])
ax[0].axhline(lim, color="r", ls="--", lw=0.9, label="IC 95 % de blancura")
ax[0].axhline(-lim, color="r", ls="--", lw=0.9)
ax[0].set(xlabel="Retardo [ms]", ylabel=r"$\rho[m]$",
          title="Autocorrelacion normalizada del ruido")
ax[0].legend()

# todos los retardos con la escala de los datos: el IC queda casi pegado al cero,
# y eso es justamente lo que hay que ver
ax[1].plot(lags_ms[1:], rho[1:], lw=0.9, color=COLOR["ruido"])
ax[1].axhline(0, color="k", lw=0.6)
ax[1].axhline(lim, color="r", ls="--", lw=0.9, label="IC 95 % de blancura")
ax[1].axhline(-lim, color="r", ls="--", lw=0.9)
ax[1].set(xlabel="Retardo [ms]", ylabel=r"$\rho[m]$",
          title=f"Todos los retardos hasta {lags_ms[-1]:.1f} ms (sin el 0)")
ax[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

for m_ in (1, 5, 50, 400):
    print(f"rho[{m_}] ({lags_ms[m_]:.2f} ms) = {rho[m_]:+.3f}")
neg = np.nonzero(rho[1:] < 0)[0] + 1
if neg.size:
    print(f"Primer cambio de signo a los {lags_ms[neg[0]]:.2f} ms; minimo {rho[1:].min():+.3f} "
          f"a los {lags_ms[1 + np.argmin(rho[1:])]:.2f} ms")
print(f"IC 95 % de blancura: +-{lim:.2e}")
print(f"Retardos fuera del IC 95 %: {100*np.mean(np.abs(rho[1:]) > lim):.1f} % "
      f"(seria ~5 % si fuese blanco)")
print(f"Retardos con |rho| > 0.01 : {np.sum(np.abs(rho[1:]) > 0.01)} de {len(rho)-1}")

*Figura 5. Autocorrelación normalizada del ruido: los primeros 60 retardos (izquierda) y todos hasta 16.7 ms (derecha). En rojo, la banda de confianza del 95 % para ruido blanco ($\pm1.63\times10^{-3}$), que a esta escala casi se pega al cero. El 98.5 % de los retardos queda afuera de esa banda, y $\rho[m]$ oscila: cambia de signo a los 4.63 ms y llega a −0.079 a los 7.08 ms, coherente con las líneas angostas de la Figura 4.*

**El ruido no es blanco.** La autocorrelación no se parece a una delta: $\rho[1] = 0.538$, a los 2.08 ms todavía $\rho[50] = 0.128$, y el 98.5 % de los retardos queda fuera del intervalo de $\pm1.63\times10^{-3}$. La PSD dice lo mismo: cae 21.6 dB entre 100 Hz y 11 kHz, o sea que domina la baja frecuencia. Es lo esperable en la cadena de un acelerómetro piezoeléctrico (ruido $1/f$ del amplificador de carga, deriva, microfonía del cable) y, con el sensor sobre un escritorio, también por la vibración del edificio.

Tampoco es solo ruido de banda ancha. $\rho[m]$ oscila (cambia de signo por primera vez a los 4.63 ms y llega a $-0.079$ a los 7.08 ms) y la PSD mantiene líneas angostas por debajo de 1 kHz aun promediando por mediana; las más marcadas están en 82.0 y 99.6 Hz (impresas junto a la Figura 4). No identificamos su origen. Con el sensor sobre un escritorio, lo más probable es vibración del edificio que llega por el mueble e interferencia eléctrica; la línea de 99.6 Hz está a menos de una resolución ($\Delta f$ = 1.46 Hz) de 100 Hz, el doble de la frecuencia de red. Por eso este piso es una cota superior del ruido propio del sensor, no el ruido propio en sí.

En resumen, el piso de referencia es coloreado, tiene algunas líneas por debajo de 1 kHz y su amplitud es muy baja: $\text{RMS} = 2.369\times10^{-3}$ g, entre 36.8 y 64.4 dB por debajo de M2-C1 según la banda (se imprime en el paso 4). No limita el análisis, porque las bandas que nos interesan están en los kHz, donde el piso es liso, pero define contra qué medimos.

`PSD_RUIDO` queda guardado como piso de referencia. Lo usamos para (a) verificar que los picos espectrales del paso 4 son reales, (b) descartar bandas sin energía en el kurtograma del paso 5 y (c) validar el rango dinámico en los indicadores de salud del paso 9.

---

# 3. Indicadores en el dominio temporal

Calculamos cuatro escalares clásicos sobre la señal ya preparada (sin continua ni deriva):

$$
\text{RMS} = \sqrt{\frac{1}{N}\sum_n x^2[n]}
\qquad
C_f = \frac{\max_n |x[n]|}{\text{RMS}}
$$

$$
\text{Kurt} = \frac{\frac{1}{N}\sum_n (x[n]-\bar x)^4}{\left[\frac{1}{N}\sum_n (x[n]-\bar x)^2\right]^2} - 3
\qquad
\text{Skew} = \frac{\frac{1}{N}\sum_n (x[n]-\bar x)^3}{\left[\frac{1}{N}\sum_n (x[n]-\bar x)^2\right]^{3/2}}
$$

El RMS mide la energía total de la vibración. Es el indicador de la norma (ISO 10816) y detecta bien desbalanceo, desalineación o soltura, que son fallas distribuidas y de gran amplitud.

El factor de cresta $C_f$ mide cuán picuda es la señal. Para una sinusoide vale $\sqrt2$, y para ruido gaussiano crece despacio con $N$ (con el $N$ de este trabajo, un gaussiano simulado da 4.73; se imprime abajo).

La kurtosis (acá en su forma de exceso, con el $-3$) es el cuarto momento normalizado. Vale $0$ para una gaussiana y crece cuando aparecen eventos de gran amplitud, que es justo lo que produce una falla localizada: un impacto en cada paso por el defecto y silencio entre medio.

La asimetría (skewness) debería dar $\approx 0$; si se aparta mucho puede indicar rozamiento en un solo sentido o problemas de montaje.

Lo central de este paso es la diferencia entre RMS y kurtosis: **una falla localizada incipiente cambia la forma de la señal mucho antes que su energía**. En estos datos se ve muy claro.

In [ ]:
filas = []
for clave, d in CH.items():
    ind = indicadores_tiempo(d["x"])
    ind = {"canal": clave, "descripcion": d["desc"], **ind}
    filas.append(ind)

tabla_t = pd.DataFrame(filas).set_index("canal")
display(tabla_t.style.format({"RMS": "{:.4f}", "Pico": "{:.3f}", "Cresta": "{:.2f}",
                              "Kurtosis": "{:.3f}", "Skewness": "{:+.3f}"}))

r1, r2 = tabla_t.loc["M2-C1"], tabla_t.loc["M2-C2"]
print(f"Comparacion dentro del MISMO equipo (Vent. 2):")
print(f"  RMS      : C1 = {r1['RMS']:.3f} g   vs   C2 = {r2['RMS']:.3f} g"
      f"   -> C2 es {r2['RMS']/r1['RMS']:.2f}x MAYOR")
print(f"  Kurtosis : C1 = {r1['Kurtosis']:.3f}    vs   C2 = {r2['Kurtosis']:.3f}"
      f"   -> C1 es el impulsivo")
z = np.random.default_rng(2543).standard_normal(len(CH["M2-C1"]["x"]))
print(f"\nReferencia: factor de cresta de ruido gaussiano simulado con el mismo N "
      f"({len(z):,} muestras): {np.abs(z).max() / z.std():.2f}")

El canal C2 del ventilador 2 tiene más del doble de RMS que el C1, pero una kurtosis prácticamente nula. Si la alarma se basara en el nivel global de vibración, que es el criterio de la norma y el más usado, **se señalaría el canal equivocado**.

La explicación física es que el lado libre soporta el ventilador y recibe más vibración sincrónica (desbalanceo del rotor, paso de álabes). Esa vibración es grande pero gaussiana y estacionaria, así que su kurtosis es nula. El lado acople tiene menos energía total, pero tiene un tren de impactos que la kurtosis sí ve. Por esto mismo hace falta todo lo que sigue en el notebook.

In [ ]:
# Los indicadores como variables aleatorias: una realizacion por ventana de 1 s.
kurt_v, rms_v_ch, claves = {}, {}, list(CH)
for clave, d in CH.items():
    V = segmentar(d["x"], d["fs"], t_seg=T_VENTANA)
    kurt_v[clave] = stats.kurtosis(V, axis=1, bias=False)
    rms_v_ch[clave] = np.sqrt(np.mean(V ** 2, axis=1))

fig, ax = plt.subplots(1, 2, figsize=(12.5, 3.8))
colores = [COLOR[CH[k]["rol"]] for k in claves]

bp = ax[0].boxplot([kurt_v[k] for k in claves], tick_labels=claves, patch_artist=True,
                   medianprops={"color": "k"})
for caja, c in zip(bp["boxes"], colores):
    caja.set_facecolor(c)
    caja.set_alpha(0.65)
ax[0].axhline(0, color="k", lw=0.8, ls=":")
# el glitch del ruido llega a kurtosis ~12 y aplastaria el resto: se recorta el eje
_kmax = max(np.percentile(kurt_v[k], 99) for k in claves if k != "RUIDO")
ax[0].set(ylabel="Kurtosis (exceso)", ylim=(-0.4, _kmax * 1.6),
          title=f"Kurtosis por ventana de {T_VENTANA:.0f} s "
                f"(RUIDO se sale de escala: max {kurt_v['RUIDO'].max():.0f})")

bp = ax[1].boxplot([rms_v_ch[k] for k in claves], tick_labels=claves, patch_artist=True,
                   medianprops={"color": "k"})
for caja, c in zip(bp["boxes"], colores):
    caja.set_facecolor(c)
    caja.set_alpha(0.65)
ax[1].set(ylabel="RMS [g]", yscale="log", title=f"RMS por ventana de {T_VENTANA:.0f} s")
fig.tight_layout()
plt.show()

print("Kurtosis por ventana: media +- desvio")
for k in claves:
    print(f"  {k:<6}: {kurt_v[k].mean():+.3f} +- {kurt_v[k].std():.3f}   "
          f"(min {kurt_v[k].min():+.2f}, max {kurt_v[k].max():+.2f})")

*Figura 6. Distribución por ventanas de 1 s de la kurtosis (izquierda; el eje está recortado porque el ruido llega a 12) y del RMS (derecha, escala logarítmica). Las dos cajas ordenan los canales distinto: M2-C1 tiene la kurtosis más alta (media +1.067) con un RMS intermedio, y M2-C2 tiene el RMS más alto con kurtosis nula (media −0.009). Los puntos sueltos de RMS bajo en M1-C2 son las ventanas finales que se ven en la Figura 2.*

In [ ]:
maq = [k for k in claves if k != "RUIDO"]     # solo canales de maquina
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

# --- evolucion temporal de la kurtosis ---
for clave in claves:
    ax[0].plot(np.arange(len(kurt_v[clave])) * T_VENTANA, kurt_v[clave], ".-", ms=4,
               lw=0.9, color=COLOR[CH[clave]["rol"]], alpha=0.85, label=clave)
ax[0].axhline(0, color="k", lw=0.8, ls=":")
ax[0].set(xlabel="Tiempo [s]", ylabel="Kurtosis (exceso)", ylim=(-0.4, _kmax * 1.6),
          title=f"Kurtosis ventana a ventana ({T_VENTANA:.0f} s)")
ax[0].legend(fontsize=8, ncol=2)

# --- el plano RMS - kurtosis, solo canales de maquina ---
for clave in maq:
    ax[1].scatter(rms_v_ch[clave], kurt_v[clave], s=20, alpha=0.75, edgecolors="none",
                  color=COLOR[CH[clave]["rol"]], label=clave)
ax[1].annotate("mas energia,\nmenos impulsivo", xy=(3.35, 0.22), fontsize=8,
               color="#2471a3", ha="center")
ax[1].annotate("menos energia,\nMAS impulsivo", xy=(1.70, 1.75), fontsize=8,
               color=COLOR["falla"], ha="center")
ax[1].set(xlabel="RMS de la ventana [g]", ylabel="Kurtosis (exceso)",
          xlim=(0.8, 4.2), ylim=(-0.4, _kmax * 1.6),
          title="Plano RMS - kurtosis (canales de maquina)")
ax[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

*Figura 7. Izquierda: kurtosis de cada canal ventana a ventana. Derecha: las ventanas de los canales de máquina en el plano RMS–kurtosis. Los grupos se separan en vertical. M2-C1 queda arriba, con un RMS (1.68 g) que cae entre los de los canales sanos (1.36 y 2.92 g), así que un umbral sobre el RMS no lo aísla.*

La kurtosis alta de M2-C1 es **persistente**. Su mínimo por ventana (+0.40) supera el máximo de M2-C2 y de M1-C1 (+0.16 y +0.12), y en la traza temporal se mantiene positiva en todas las ventanas, así que no es una ventana atípica. M1-C2 se superpone en parte (su máximo es +1.70). El ruido, en cambio, tiene kurtosis alta pero con muchísima dispersión (máximo +12.22): es el *glitch* del paso 2 concentrado en pocas ventanas, que se ve como picos aislados sobre una base nula. La diferencia entre un corrimiento de toda la distribución y unos pocos valores extremos es lo que separa una falla de un artefacto de medición, y es la razón para usar estadísticos por ventana y no un único valor global.

En el plano RMS–kurtosis los grupos se separan en vertical (kurtosis) y no en horizontal (RMS). M2-C1 queda arriba, con un RMS intermedio entre los de los canales sanos (1.68 g, entre 1.36 y 2.92 g). Un criterio de nivel global, que equivale a proyectar sobre el eje horizontal, no lo aísla.

---

# 4. Espectro clásico: Welch y modelo AR

### Qué esperamos ver

En el espectro de aceleración de una máquina rotativa aparecen tres cosas. Las componentes sincrónicas: $1\times f_r$ (desbalanceo), $2\times f_r$ (desalineación) y el paso de álabes $N_{\text{álabes}}\times f_r$, todas múltiplos enteros de $f_r$. Las componentes de falla de rodamiento, en múltiplos no enteros de $f_r$ y mucho más chicas. Y las resonancias estructurales excitadas por los impactos, que se ven como elevaciones anchas (*humps*) en alta frecuencia.

Lo importante es que **la falla de rodamiento casi nunca se ve directamente en este espectro**. La energía de cada impacto se reparte por toda la banda de resonancia y la línea en BPFI queda enterrada. Lo que sí se ven son las resonancias, como *humps* en los kHz. Así que este paso sirve para dos cosas: descartar que el problema sea sincrónico y ubicar esas resonancias, que como veremos en el paso 5 no coinciden con la mejor banda para demodular.

### Por qué Welch y no el periodograma

El periodograma simple,

$$\hat{P}_{per}(f) = \frac{1}{f_s N}\left|\sum_{n=0}^{N-1} x[n]e^{-j2\pi fn/f_s}\right|^2,$$

es asintóticamente insesgado pero inconsistente: su varianza tiende a $P^2(f)$ y no baja al crecer $N$. Un registro más largo da más resolución, no más precisión, y la curva sigue igual de erizada. Welch cambia resolución por varianza: al promediar $K$ segmentos independientes, la varianza cae como $1/K$.

La primera figura lo muestra sobre la señal real. La segunda muestra el otro compromiso, el de las ventanas. Como $x_w[n] = x[n]w[n]$ implica $X_w(f) = X(f) * W(f)$, lo que vemos es el espectro verdadero convolucionado con la transformada de la ventana. La rectangular tiene el lóbulo principal más angosto (mejor resolución) pero lóbulos laterales altos, que desparraman la energía de los picos fuertes por todo el eje (*leakage*). Hann y Blackman ensanchan el lóbulo principal a cambio de bajar los laterales.

In [ ]:
x1 = CH["M2-C1"]["x"]

fig, ax = plt.subplots(1, 2, figsize=(13.5, 4.2))

# --- periodograma vs Welch ---
f_per, P_per = signal.periodogram(x1, FS_ACEL, window="boxcar")
ax[0].semilogy(f_per[1:], P_per[1:], lw=0.3, color="#bdc3c7",
               label=f"periodograma (K=1, $\\Delta f$={FS_ACEL/len(x1):.3f} Hz)")
for L, c, lw in [(2 ** 14, COLOR["falla"], 0.7), (2 ** 10, "#e67e22", 1.9)]:
    fw, Pw = psd_welch(x1, FS_ACEL, nperseg=L)
    K = len(x1) // (L // 2) - 1
    ax[0].semilogy(fw[1:], Pw[1:], lw=lw, color=c,
                   label=f"Welch L={L} (K={K}, $\\Delta f$={FS_ACEL/L:.2f} Hz)")
ax[0].set(xlabel="Frecuencia [Hz]", ylabel=r"PSD [$g^2$/Hz]", xlim=(1000, 5000),
          title="Periodograma vs. Welch: reduccion de varianza")
ax[0].legend(fontsize=8, loc="lower right")

# --- las ventanas mismas: lobulo principal vs lobulos laterales ---
L_w = 256
lobulos = {}
for vent, c in [("boxcar", "#8e44ad"), ("hann", COLOR["falla"]), ("blackman", "#16a085")]:
    w = signal.get_window(vent, L_w)
    W = np.abs(np.fft.rfft(w, 1 << 16))
    Wd = 20 * np.log10(W / W.max() + 1e-16)
    bins = np.arange(len(Wd)) * L_w / (1 << 16)      # en "bines de DFT"
    pk, _ = signal.find_peaks(Wd)
    lobulos[vent] = Wd[pk[0]]
    ax[1].plot(bins, Wd, lw=1.1, color=c,
               label=f"{vent} (1er lobulo lateral {Wd[pk[0]]:.0f} dB)")
ax[1].set(xlabel="Frecuencia [bines de DFT]", ylabel=r"$|W(f)|$ [dB]",
          xlim=(0, 8), ylim=(-120, 5),
          title="Ventanas espectrales: resolucion vs. leakage")
ax[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

# --- cuantificacion de la varianza del estimador ---
# No alcanza con el desvio de P sobre una banda: eso mide la FORMA del espectro, no el
# ruido del estimador. Se divide P por una version suavizada de si misma (filtro de
# mediana) y se mide la dispersion del cociente, que aisla el "erizado".
from scipy import ndimage


def jitter(f, P, lo=2000, hi=3000, n_med=41):
    suave = ndimage.median_filter(P, size=n_med, mode="nearest")
    m = (f > lo) & (f < hi)
    return float((P[m] / suave[m]).std())


j_per = jitter(f_per, P_per)
print(f"Erizado del estimador en 2-3 kHz (desvio de P / suavizado de P):")
print(f"  periodograma   : {j_per:.3f}   (K=1)")
for L in [2 ** 10, 2 ** 14]:
    fw, Pw = psd_welch(x1, FS_ACEL, nperseg=L)
    K = len(x1) // (L // 2) - 1
    j = jitter(fw, Pw, n_med=11 if L < 4096 else 41)
    print(f"  Welch L={L:<6}: {j:.3f}   (K={K}, reduccion {j_per/j:.1f}x, "
          f"cota ideal 1/sqrt(K) = {1/np.sqrt(K):.3f})")

fw, Pw = psd_welch(x1, FS_ACEL, nperseg=2 ** 14)
mh = (fw > 1000) & (fw < 5000)
print(f"\nMaximo de la PSD de Welch (L = 2^14) entre 1 y 5 kHz: {fw[mh][np.argmax(Pw[mh])]:.0f} Hz")
print("Primer lobulo lateral: " + ", ".join(f"{v} {l:.1f} dB" for v, l in lobulos.items()))

print("\nEfecto de la ventana sobre ESTA señal (PSD media entre 1x y 2x de fr):")
vals_v = []
for vent in ["boxcar", "hann", "blackman"]:
    fw, Pw = psd_welch(x1, FS_ACEL, nperseg=2 ** 16, ventana=vent)
    mw = (fw > 20) & (fw < 30)
    vals_v.append(Pw[mw].mean())
    print(f"  {vent:<9}: {Pw[mw].mean():.4e}")
print(f"  diferencia maxima entre ventanas: {100 * (max(vals_v) - min(vals_v)) / min(vals_v):.1f} %")

*Figura 8. Izquierda: periodograma de los 60 s de M2-C1 (gris) contra Welch con $L = 2^{10}$ y $2^{14}$, entre 1 y 5 kHz. El periodograma se dispersa varios órdenes de magnitud y Welch reduce esa dispersión entre 3.6 y 7.3 veces; sobre Welch aparece el hump, con máximo en 3924 Hz. Derecha: transformada de las ventanas rectangular, Hann y Blackman, en bines de DFT. El primer lóbulo lateral baja de −13.3 dB (rectangular) a −31.5 dB (Hann) y −96.2 dB (Blackman), con un lóbulo principal dos y tres veces más ancho.*

El periodograma (gris) es una nube que cubre varios órdenes de magnitud alrededor del valor verdadero, y lo sigue siendo aunque usemos el registro completo de 1.44 millones de muestras, porque su varianza no depende de $N$. Welch promedia segmentos y reduce el erizado entre 3.6 y 7.3 veces según $L$. Sobre esa curva suave sí se puede decidir si una elevación es una resonancia o ruido de estimación, y ahí aparece claro el *hump*, con su máximo en 3924 Hz.

La reducción medida no llega a la cota ideal $1/\sqrt{K}$. No es por el solape: con Hann al 50 % los segmentos vecinos quedan casi descorrelacionados y el $K$ efectivo es prácticamente el nominal. Lo que pasa es que lo que el indicador cuenta como erizado incluye estructura espectral real (armónicos, bandas laterales, la forma del espectro dentro de la ventana del suavizado) que el filtro de mediana no saca, y que se resuelve más cuanto más fina es la resolución. Por eso $L = 2^{14}$ da más erizado que $L = 2^{10}$. La cota vale para un proceso de espectro liso, y esta señal no lo es.

En la figura de la derecha se ve el compromiso de ventaneo: la rectangular tiene el lóbulo principal más angosto, pero su primer lóbulo lateral está solo a −13.3 dB; Hann llega a −31.5 dB y Blackman a −96.2 dB, con un lóbulo principal dos y tres veces más ancho.

Ahora, en estos datos **el ventaneo casi no cambia nada**. Sobre la señal real, las tres ventanas dan PSDs que difieren en un 2.9 % (el número está impreso arriba). El *leakage* no es un problema porque la señal es de banda ancha: no hay ninguna componente angosta tan dominante como para que sus lóbulos laterales tapen a las vecinas. El ventaneo es crítico cuando se quiere ver una componente débil al lado de una muy fuerte, y este no es el caso. Usamos Hann en todo el notebook porque es la elección estándar, no porque acá cambie el resultado.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.6))
for clave in ["M2-C1", "M2-C2", "M1-C1"]:
    d = CH[clave]
    f, P = psd_welch(d["x"], d["fs"], nperseg=NPERSEG_PSD)
    ax.loglog(f[1:], P[1:], lw=0.7, color=COLOR[d["rol"]] if clave != "M1-C1" else COLOR["ref"],
              alpha=0.9, label=f"{clave} — {d['desc']}")
ax.loglog(F_RUIDO[1:], PSD_RUIDO[1:], lw=0.9, color="k", ls=":",
          label="piso de ruido del sensor")

marcar(ax, [k * fr for k in range(1, 11)],
       [f"{k}x" if k in (1, 2, 4, 6, 8, 10) else None for k in range(1, 11)],
       color="#8e44ad")
ax.set(xlabel="Frecuencia [Hz]", ylabel=r"PSD [$g^2$/Hz]",
       title="PSD de Welch (log-log): señales de maquina vs. piso de ruido",
       xlim=(5, FS_ACEL / 2))
ax.legend(loc="lower left")
fig.tight_layout()
plt.show()

# margen sobre el ruido, por banda
print("Margen señal/ruido por banda (M2-C1):")
f, P1 = psd_welch(CH["M2-C1"]["x"], FS_ACEL, nperseg=NPERSEG_PSD)
Pr = np.interp(f, F_RUIDO, PSD_RUIDO)
for lo, hi in [(10, 100), (100, 500), (500, 2000), (2000, 5000), (5000, 9000), (9000, 11800)]:
    m = (f >= lo) & (f < hi)
    print(f"  {lo:>5}-{hi:<5} Hz : {10*np.log10(P1[m].mean()/Pr[m].mean()):5.1f} dB")

*Figura 9. PSD de Welch ($L = 2^{14}$) de tres canales de máquina y del piso de ruido (punteado), con los armónicos de $f_r$ marcados. En M2-C1 la distancia al piso va de 36.8 dB (10–100 Hz) a 64.4 dB (2–5 kHz). El hump de resonancia en los kHz aparece en los tres canales, y es más alto en M2-C2 y M1-C1 que en M2-C1, así que es de la estructura y no de la falla. En el piso se ven las líneas angostas del paso 2.*

In [ ]:
# Zoom en la region de baja frecuencia: componentes sincronicas
fig, ax = plt.subplots(figsize=(12, 4))
for clave in ["M2-C1", "M2-C2"]:
    d = CH[clave]
    f, P = psd_welch(d["x"], d["fs"], nperseg=2 ** 16)
    ax.semilogy(f, P, lw=0.8, color=COLOR[d["rol"]], alpha=0.9, label=clave)
marcar(ax, [k * fr for k in range(1, 13)],
       [f"{k}x fr" for k in range(1, 13)], color="#8e44ad")
marcar(ax, [TEO["BPFI"], TEO["BPFO"]], ["BPFI teorico", "BPFO teorico"],
       color=COLOR["falla"], ls="-")
ax.set(xlabel="Frecuencia [Hz]", ylabel=r"PSD [$g^2$/Hz]", xlim=(0, 260),
       title="Zoom de baja frecuencia: armonicos de giro y frecuencias de falla teoricas")
ax.legend()
fig.tight_layout()
plt.show()

*Figura 10. Zoom de 0 a 260 Hz de M2-C1 y M2-C2 ($L = 2^{16}$), con los armónicos de $f_r$ y las frecuencias teóricas de falla: BPFO (136.2 Hz) y BPFI (180.4 Hz). El espectro está dominado por múltiplos enteros de $f_r$ y en BPFI no hay nada; el pico cerca de BPFO es el de $8\times f_r$, sincrónico. La falla no aparece como línea en el espectro de la señal cruda.*

Como esperábamos, en baja frecuencia dominan los múltiplos enteros de $f_r$ y no hay nada reconocible en BPFI ni en BPFO (el pico vecino a BPFO es el de $8\times f_r$, sincrónico). **El espectro de aceleración convencional no alcanza para este diagnóstico.** Y en los kHz (Figura 9) el *hump* de resonancia está en los tres canales, en M2-C2 y M1-C1 incluso más alto que en M2-C1, así que es una resonancia de la estructura y no una firma de la falla.

### Estimación paramétrica: modelo AR y Yule-Walker

Esta sección es una demostración y ningún paso posterior usa su resultado; el diagnóstico no pasa por acá. La incluimos para comparar un estimador paramétrico con Welch en el caso donde el paramétrico tiene ventaja, y para mostrar por qué tampoco así aparece la falla.

Un modelo autorregresivo de orden $p$ describe la señal como ruido blanco filtrado por un sistema todo-polos:

$$x[n] = \sum_{k=1}^{p} a_k\,x[n-k] + w[n],
\qquad
\hat{P}_{AR}(f) = \frac{\sigma_w^2/f_s}{\left|1 - \sum_{k=1}^{p} a_k e^{-j2\pi f k/f_s}\right|^2}$$

Los coeficientes salen de las ecuaciones de Yule-Walker, que los relacionan con la autocorrelación del proceso:

$$
\begin{bmatrix}
r[0] & r[1] & \cdots & r[p-1]\\
r[1] & r[0] & \cdots & r[p-2]\\
\vdots & & \ddots & \vdots\\
r[p-1] & r[p-2] & \cdots & r[0]
\end{bmatrix}
\begin{bmatrix} a_1\\ a_2\\ \vdots\\ a_p\end{bmatrix}
=
\begin{bmatrix} r[1]\\ r[2]\\ \vdots\\ r[p]\end{bmatrix}
$$

La matriz es Toeplitz y se resuelve en $O(p^2)$ con Levinson-Durbin (`scipy.linalg.solve_toeplitz`). El AR no ventanea, así que no tiene *leakage* ni está atado a $\Delta f = f_s/L$, pero impone un modelo todo-polos. Trabajamos sobre la señal diezmada a 1500 Hz y con 2 s de registro, porque con 60 s a 24 kHz Welch promedia tantos segmentos que no hay nada que comparar.

In [ ]:
def ar_yule_walker(r, p):
    # Resuelve Yule-Walker a partir de la autocorrelacion r. Devuelve (a, sigma2)
    # con la convencion x[n] = sum a_k x[n-k] + w[n].
    a = linalg.solve_toeplitz((r[:p], r[:p]), r[1:p + 1])
    return a, float(r[0] - a @ r[1:p + 1])


def psd_ar(a, sigma2, fs, nfft=4096):
    w, h = signal.freqz(np.sqrt(sigma2), np.r_[1.0, -a], worN=nfft, fs=fs)
    return w, np.abs(h) ** 2 / fs


Q = 16                                   # factor de diezmado
T_CORTO = 2.0                            # s de registro para la comparacion
xd = signal.decimate(CH["M2-C1"]["x"], Q, ftype="fir", zero_phase=True)
fsd = FS_ACEL / Q
xs = xd[:int(fsd * T_CORTO)]
N = len(xs)
print(f"Señal diezmada: fs = {fsd:.0f} Hz, registro de {T_CORTO} s -> N = {N} muestras")

# --- seleccion de orden ---
p_max = 300
r_xs = autocorr(xs, p_max)
ordenes = np.arange(5, p_max + 1, 5)
sig2 = np.array([ar_yule_walker(r_xs, int(p))[1] for p in ordenes])
aic = N * np.log(sig2) + 2 * ordenes
fpe = sig2 * (N + ordenes) / (N - ordenes)

fig, ax = plt.subplots(1, 2, figsize=(12.5, 3.4))
ax[0].plot(ordenes, aic, ".-", color=COLOR["falla"])
ax[0].set(xlabel="Orden p", ylabel="AIC", title="Criterio de informacion de Akaike")
ax[1].semilogy(ordenes, sig2, ".-", color=COLOR["falla"])
ax[1].set(xlabel="Orden p", ylabel=r"$\sigma_w^2$",
          title="Varianza residual del modelo")
fig.tight_layout()
plt.show()

p_aic = int(ordenes[np.argmin(aic)])
print(f"AIC minimo en p = {p_aic}   FPE minimo en p = {ordenes[np.argmin(fpe)]}")

# convencion habitual: modelos con delta-AIC <= 2 son indistinguibles entre si
meseta = ordenes[aic <= aic.min() + 2]
print(f"Ordenes con delta-AIC <= 2 (indistinguibles del optimo): {list(meseta)}")
print(f"El optimo usa {p_aic} parametros para {N} muestras: "
      f"1 cada {N/p_aic:.0f}")

# --- criterio de la rodilla: primer orden con mejora relativa < 0.5 % ---
mejora = -np.diff(np.log(sig2)) / np.diff(ordenes) * 5     # por cada +5 de orden
p_opt = int(ordenes[1:][np.argmax(mejora < 0.005)])
print(f"Criterio de la rodilla (mejora relativa < 0.5 % cada +5 ordenes): p = {p_opt}")

*Figura 11. AIC (izquierda) y varianza residual $\sigma_w^2$ (derecha) del modelo AR según el orden, sobre 2 s de M2-C1 diezmada a 1500 Hz. El AIC baja hasta una meseta entre $p \approx 100$ y 160 y tiene su mínimo en $p = 215$. La varianza residual baja sin una rodilla clara; con el criterio de mejora menor al 0.5 % queda en $p = 40$. La Figura 12 decide entre los dos.*

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.4))
for L, est in [(256, "Welch L=256"), (1024, "Welch L=1024")]:
    if L <= N:
        fw, Pw = signal.welch(xs, fsd, nperseg=L, noverlap=L // 2)
        ax.semilogy(fw, Pw, lw=1.0, alpha=0.75,
                    label=f"{est} ($\\Delta f$ = {fsd/L:.1f} Hz)")

for p, ls, etq in [(20, ":", "muy suavizado"), (p_opt, "--", "rodilla"),
                   (p_aic, "-", "AIC")]:
    a, s2 = ar_yule_walker(r_xs, int(p))
    fa, Pa = psd_ar(a, s2, fsd)
    ax.semilogy(fa, Pa, ls, lw=1.4, label=f"AR(p={p}) — {etq}")

marcar(ax, [k * fr for k in range(1, 9)], [f"{k}x" for k in range(1, 9)], color="#8e44ad")
ax.set(xlabel="Frecuencia [Hz]", ylabel=r"PSD [$g^2$/Hz]", xlim=(0, 200),
       title=f"Welch vs. AR sobre {T_CORTO} s de señal diezmada a {fsd:.0f} Hz")
ax.legend(ncol=2, fontsize=8)
fig.tight_layout()
plt.show()

*Figura 12. Welch ($L$ = 256 y 1024) contra modelos AR de orden 20, 40 (rodilla) y 215 (AIC), sobre los mismos 2 s diezmados, de 0 a 200 Hz, con los armónicos de $f_r$ marcados (separados 16.7 Hz). Welch con $L = 256$ ($\Delta f$ = 5.9 Hz) no los resuelve; con $L = 1024$ ($\Delta f$ = 1.5 Hz) sí, pero con mucha varianza. AR(215) los resuelve con una curva suave, y AR(40) y AR(20) los borran.*

Con 2 s de registro, Welch con $L = 256$ ($\Delta f = 5.9$ Hz) no separa los armónicos de $f_r$, que están a 16.7 Hz; con $L = 1024$ ($\Delta f = 1.5$ Hz) los separa, pero con tan pocos segmentos que la estimación es ruidosa. Es el compromiso sesgo/varianza en estado puro.

Entre los dos criterios de orden, la figura favorece al AIC: AR(215) resuelve los armónicos con una curva suave y AR(40) los borra. La rodilla de $\sigma_w^2(p)$ sigue el ajuste de la parte de banda ancha, que se satura con pocos polos; los armónicos pesan poco en la varianza total y el criterio no los ve. O sea, **el orden "óptimo" depende de para qué se quiere el modelo**.

Lo que el AR no muestra es la falla, y no por falta de resolución. La falla no es una componente de la señal cruda sino una modulación: en el espectro crudo aparece como un peine de armónicos de orden alto de BPFI dentro de la banda de resonancia, desparramado por el deslizamiento. Ningún estimador del espectro crudo lo recupera, por fino que sea. Demodulando sí (paso 6).

---

# 5. Selección automática de la banda de demodulación: kurtograma rápido

### El problema

Para el análisis de envolvente hay que elegir una banda antes de demodular, y esa elección determina el resultado: si se demodula una banda sin impactos, el espectro de envolvente sale vacío aunque la falla exista. Lo habitual es elegirla a ojo sobre el *hump* de resonancia, lo cual es subjetivo y no reproducible.

### La kurtosis espectral

Antoni propone medir, para cada frecuencia, cuán impulsiva es la señal en esa banda. Si $X(t,f)$ es la envolvente compleja de la banda centrada en $f$, la kurtosis espectral es

$$K(f) \;=\; \frac{\left\langle |X(t,f)|^4 \right\rangle_t}
{\left\langle |X(t,f)|^2 \right\rangle_t^{2}} \;-\; 2$$

El $-2$ centra el indicador: para un proceso gaussiano complejo circular estacionario, $|X|^2$ es exponencial y $\mathbb{E}|X|^4 = 2\,(\mathbb{E}|X|^2)^2$, así que $K(f)=0$. Entonces, $K(f) \approx 0$ indica que en la banda hay ruido o vibración estacionaria, y $K(f) > 0$ que hay transitorios que aparecen y desaparecen, es decir, impactos.

### El kurtograma y su versión rápida

$K(f)$ depende del ancho de banda: si la banda es muy ancha, el impacto se promedia con todo lo demás, y si es muy angosta, la respuesta al impulso del filtro se alarga más que el intervalo entre impactos y los junta. El kurtograma evalúa $K(f)$ en un plano (frecuencia central, ancho de banda) y toma el máximo.

La versión rápida aprovecha que, si $X(f)$ es la FFT de toda la señal, tomar un bloque contiguo de $L$ bines y antitransformarlo da directamente la envolvente compleja diezmada de esa banda, con costo $O(L\log L)$. Recorrer todas las bandas de un nivel cuesta $O(N\log N)$ y el árbol completo de $K$ niveles, $O(K N\log N)$. Usamos el árbol *1/3-binario* de Antoni, que alterna divisiones en $2^k$ y $3\cdot 2^{k-1}$ bandas para cubrir mejor los anchos de banda.

### Dos cuidados

La kurtosis espectral es un cociente: mide **contraste, no energía**. Por eso es sensible a bandas donde casi no hay nada, y hay que acotar dónde se busca.

El primer cuidado son las bandas sin energía útil. Una banda casi vacía puede dar una kurtosis enorme solo porque se normaliza por una potencia media minúscula. En estos mismos datos está el caso extremo: sobre el registro de ruido, el kurtograma da $SK = 243.86$, 124 veces el del canal con falla, en una banda donde no hay señal. La mayor parte viene del transitorio dominante del paso 2; sacando los 2.5 s que lo rodean baja a 19.68, que igual es 10 veces más que el canal con falla. Un detector guiado por el máximo de SK diría que el registro de ruido es el más dañado de todos. Por eso exigimos que la potencia de la banda supere el piso de ruido del paso 2 por un margen mínimo.

El segundo son los bordes del espectro. Cerca de Nyquist actúa el filtro antialias de la adquisición, que atenúa y deforma la banda, así que por precaución descartamos el 10 % superior del eje. Por abajo descartamos la zona de órdenes de eje ($< 500$ Hz), que no corresponde a una resonancia estructural. En estos datos, las guardas no cambian la banda elegida para M2-C1 (se imprime abajo).

In [ ]:
def niveles_arbol(nlevel):
    # Arbol 1/3-binario: niveles con 2^k y 3*2^(k-1) bandas.
    ls = []
    for k in range(1, nlevel + 1):
        ls.append((float(k), 2 ** k))
        ls.append((k + np.log2(3) - 1, 3 * 2 ** (k - 1)))
    return sorted(ls)


def kurtograma(x, fs, nlevel=7, f_min=0.0, f_max_frac=1.0,
               psd_ruido=None, f_ruido=None, margen_db=6.0):
    # Kurtograma rapido. Devuelve la lista de niveles con su vector de SK.
    # Cada banda se obtiene extrayendo un bloque contiguo de la FFT y antitransformando:
    # eso da la envolvente compleja diezmada de la banda, sin filtrar explicitamente.
    X = np.fft.rfft(np.asarray(x, float))
    M = len(X) - 1                      # bines utiles (se excluye la continua)
    nyq = fs / 2
    filas = []
    for lvl, nb in niveles_arbol(nlevel):
        B = nyq / nb
        sk = np.full(nb, np.nan)
        pot = np.full(nb, np.nan)
        bordes = np.round(np.linspace(0, M, nb + 1)).astype(int)
        fc = (np.arange(nb) + 0.5) * B
        for j in range(nb):
            i1, i2 = bordes[j] + 1, bordes[j + 1] + 1
            if i2 - i1 < 16:            # muy pocas muestras -> estimador inestable
                continue
            if fc[j] < f_min or fc[j] > f_max_frac * nyq:
                continue
            z = np.fft.ifft(X[i1:i2])   # envolvente compleja diezmada de la banda
            a2 = np.abs(z) ** 2
            m2 = a2.mean()
            if m2 <= 0:
                continue
            pot[j] = m2
            if psd_ruido is not None:
                p_rui = np.interp(fc[j], f_ruido, psd_ruido) * B   # potencia de ruido
                if m2 < p_rui * 10 ** (margen_db / 10):
                    continue            # banda sin energia util
            sk[j] = (a2 ** 2).mean() / m2 ** 2 - 2.0
        filas.append({"lvl": lvl, "nb": nb, "B": B, "fc": fc, "sk": sk, "pot": pot})
    return filas


def mejor_banda(filas):
    mejor = None
    for r in filas:
        if np.all(np.isnan(r["sk"])):
            continue
        j = int(np.nanargmax(r["sk"]))
        if mejor is None or r["sk"][j] > mejor["SK"]:
            mejor = {"SK": float(r["sk"][j]), "fc": float(r["fc"][j]),
                     "B": float(r["B"]), "nivel": r["lvl"], "nb": r["nb"]}
    mejor["banda"] = (mejor["fc"] - mejor["B"] / 2, mejor["fc"] + mejor["B"] / 2)
    return mejor


def dibujar_kurtograma(filas, fs, ax, titulo):
    # Rasteriza los niveles sobre una grilla comun de frecuencia.
    nf = 1024
    fg = np.linspace(0, fs / 2, nf)
    img = np.full((len(filas), nf), np.nan)
    for i, r in enumerate(filas):
        idx = np.clip((fg / r["B"]).astype(int), 0, r["nb"] - 1)
        img[i] = r["sk"][idx]
    im = ax.imshow(img, aspect="auto", origin="lower", cmap="inferno",
                   extent=[0, fs / 2, -0.5, len(filas) - 0.5])
    ax.set_yticks(range(len(filas)))
    ax.set_yticklabels([f"{r['lvl']:.1f}  ({r['B']:.0f} Hz)" for r in filas], fontsize=7)
    ax.set(xlabel="Frecuencia central [Hz]", ylabel="nivel  (ancho de banda)",
           title=titulo)
    ax.grid(False)
    return im


print("kurtograma definido")

In [ ]:
NLEVEL = 7
KURT_FMIN = 500.0        # descarta la zona de ordenes de eje (no es una resonancia)
KURT_FMAXF = 0.90        # descarta el borde de Nyquist (filtro antialias)

x_falla = CH["M2-C1"]["x"]

# --- (a) el canal bajo estudio, con guardas ---
k_falla = kurtograma(x_falla, FS_ACEL, nlevel=NLEVEL, f_min=KURT_FMIN,
                     f_max_frac=KURT_FMAXF, psd_ruido=PSD_RUIDO, f_ruido=F_RUIDO)
b_falla = mejor_banda(k_falla)

# --- (b) el registro de RUIDO sin la guarda de energia: el artefacto ---
k_rui = kurtograma(CH["RUIDO"]["x"], FS_ACEL, nlevel=NLEVEL, f_min=0.0, f_max_frac=1.0)
b_rui = mejor_banda(k_rui)

fig, ax = plt.subplots(1, 2, figsize=(13.5, 4.4))
im = dibujar_kurtograma(k_falla, FS_ACEL, ax[0],
                        f"(a) M2-C1 con guardas: banda optima "
                        f"{b_falla['banda'][0]:.0f}-{b_falla['banda'][1]:.0f} Hz")
ax[0].plot(b_falla["fc"], [r["lvl"] for r in k_falla].index(b_falla["nivel"]),
           "cx", ms=12, mew=2)
fig.colorbar(im, ax=ax[0], label="SK")

im = dibujar_kurtograma(k_rui, FS_ACEL, ax[1],
                        "(b) RUIDO sin guarda de energia: SK enorme sin señal")
ax[1].plot(b_rui["fc"], [r["lvl"] for r in k_rui].index(b_rui["nivel"]),
           "cx", ms=12, mew=2)
fig.colorbar(im, ax=ax[1], label="SK")
fig.tight_layout()
plt.show()

print(f"(a) M2-C1, con guardas : SK = {b_falla['SK']:.2f} en fc = {b_falla['fc']:.0f} Hz, "
      f"B = {b_falla['B']:.0f} Hz -> banda "
      f"[{b_falla['banda'][0]:.0f}, {b_falla['banda'][1]:.0f}] Hz")
print(f"(b) RUIDO, sin guardas : SK = {b_rui['SK']:.2f} en fc = {b_rui['fc']:.0f} Hz, "
      f"B = {b_rui['B']:.0f} Hz   <-- {b_rui['SK']/b_falla['SK']:.0f} veces el SK del "
      f"canal con falla")

# robustez: el optimo de M2-C1 no depende de las guardas
b_falla_sin = mejor_banda(kurtograma(x_falla, FS_ACEL, nlevel=NLEVEL,
                                     f_min=0.0, f_max_frac=1.0))
print(f"\nM2-C1 sin guardas      : SK = {b_falla_sin['SK']:.2f} en fc = "
      f"{b_falla_sin['fc']:.0f} Hz, B = {b_falla_sin['B']:.0f} Hz "
      f"-> {'MISMA banda' if abs(b_falla_sin['fc']-b_falla['fc']) < 1 else 'banda distinta'}")

# cuanto del SK del ruido viene del transitorio dominante del paso 2: se lo saca y se
# recalcula (el empalme agrega un escalon, que si acaso SUBE la kurtosis)
t_g = t_v[i_glitch]
fuera = np.ones(len(CH["RUIDO"]["x"]), bool)
fuera[int((t_g - 1.0) * FS_ACEL):int((t_g + 1.5) * FS_ACEL)] = False
b_rui_sg = mejor_banda(kurtograma(CH["RUIDO"]["x"][fuera], FS_ACEL, nlevel=NLEVEL))
print(f"RUIDO sin {t_g - 1.0:.1f}-{t_g + 1.5:.1f} s  : SK = {b_rui_sg['SK']:.2f} en "
      f"fc = {b_rui_sg['fc']:.0f} Hz ({b_rui_sg['SK'] / b_falla['SK']:.0f} veces el del canal con falla)")

BANDA = b_falla["banda"]

*Figura 13. Kurtograma rápido: kurtosis espectral según la frecuencia central (eje x) y el ancho de banda (eje y, un nivel por fila). (a) M2-C1 con las guardas; las zonas blancas son las bandas descartadas y la cruz marca el máximo: SK = 1.97 en 1500–3000 Hz. (b) Registro de ruido sin la guarda de energía: SK = 243.86 en una banda de 62 Hz alrededor de 531 Hz, 124 veces el del canal con falla, en una zona sin señal.*

En (a), el kurtograma de M2-C1 elige la banda de 1500 a 3000 Hz. No es el *hump* de la Figura 9, que tiene su máximo en 3924 Hz y también está en los canales sanos: el kurtograma no busca dónde hay más energía sino dónde hay más impulsividad. Con esto la banda de demodulación queda justificada con un número; a ojo sobre el *hump* habríamos elegido otra.

En (b) está el artefacto. El registro de ruido, sin la guarda de energía, llega a un SK 124 veces mayor que el del canal que sí tiene la falla, en una banda angosta de baja frecuencia sin señal. La mayor parte viene del transitorio del paso 2 (sin él baja a 19.68), pero igual queda 10 veces por encima del canal con falla. Esto muestra que **el máximo de SK no es un indicador de falla**: es un indicador de contraste, y el contraste puede venir tanto de la ausencia de señal como de su presencia.

Para M2-C1, además, las guardas no cambian el resultado: la banda óptima es la misma con y sin restricciones. O sea que las guardas no empujan la respuesta hacia donde queremos, solo descartan regiones donde la pregunta no tiene sentido.

In [ ]:
# El mismo procedimiento aplicado a todos los canales, para comparar.
KURT = {}
for clave, d in CH.items():
    filas = kurtograma(d["x"], d["fs"], nlevel=NLEVEL, f_min=KURT_FMIN,
                       f_max_frac=KURT_FMAXF, psd_ruido=PSD_RUIDO, f_ruido=F_RUIDO)
    KURT[clave] = mejor_banda(filas)

res = pd.DataFrame(KURT).T[["SK", "fc", "B"]]
res["banda [Hz]"] = [f"{v['banda'][0]:.0f} - {v['banda'][1]:.0f}" for v in KURT.values()]
res.columns = ["SK max", "fc [Hz]", "B [Hz]", "banda [Hz]"]
display(res.style.format({"SK max": "{:.2f}", "fc [Hz]": "{:.0f}", "B [Hz]": "{:.0f}"}))

Una kurtosis espectral alta **no alcanza por sí sola para diagnosticar una falla**, y la tabla de arriba lo deja claro: los canales de referencia M1-C1 y M1-C2 llegan a un $SK$ de $4.46$ y $7.53$, más que el $1.97$ del canal que tiene la falla. El kurtograma solo dice que en esa banda hay algo no estacionario. Puede ser una falla de rodamiento, pero también rozamiento, cavitación, holgura, flujo turbulento o un artefacto de la cadena de medición.

Lo que convierte ese indicio en diagnóstico es el paso siguiente: demodular la banda y ver si la envolvente tiene una periodicidad compatible con alguna frecuencia de paso del rodamiento, con armónicos y bandas laterales. El kurtograma dice dónde mirar y el espectro de envolvente dice qué hay. Si ordenáramos los canales por $SK$ tendríamos otra vez el ranking equivocado, igual que con el RMS en el paso 3.

---

# 6. Análisis de envolvente

Al conversar con los profesionales de mantenimiento predictivo, me indicaron que en la industria se utiliza un análisis de **envolvente**, para diagnosticar las fallas. A raíz de esa conversación, se llevó a cabo la siguiente investigación.

### El modelo de señal

Cada vez que un rodillo pasa por un defecto en la pista se produce un impacto muy corto. La estructura responde a cada impacto con su respuesta impulsiva $h(t)$, una oscilación amortiguada a la frecuencia de resonancia $f_{\text{res}}$. La señal medida es aproximadamente

$$x(t) \;\approx\; \sum_{i} A_i\; h\!\left(t - i\,T_d - \tau_i\right) \;+\; n(t)$$

donde $T_d = 1/f_d$ es el intervalo entre impactos ($f_d$ = frecuencia de paso del defecto), $\tau_i$ es una pequeña fluctuación aleatoria por el deslizamiento de los rodillos, y $A_i$ modula la amplitud según dónde esté el defecto respecto de la zona de carga.

Es decir, la información de la falla está en la envolvente y no en la portadora. La portadora es $f_{\text{res}}$ (kHz) y la modulante es $f_d$ (decenas o centenas de Hz).

### Demodulación por transformada de Hilbert

Para obtener la envolvente, se utiliza la transformada de Hilbert: técnica utilizada en demodulación de AM. Se construye la señal analítica a partir de la señal filtrada $x_b(t)$:

$$z(t) = x_b(t) + j\,\mathcal{H}\{x_b\}(t),
\qquad
\mathcal{H}\{x_b\}(t) = \frac{1}{\pi}\,\text{v.p.}\!\!\int_{-\infty}^{\infty}
\frac{x_b(\tau)}{t-\tau}\,d\tau$$

En frecuencia, $\mathcal{H}$ es un filtro de módulo unitario que desfasa $-90^\circ$ las frecuencias positivas y $+90^\circ$ las negativas, así que $Z(f)$ conserva solo las frecuencias positivas, duplicadas. La envolvente instantánea es $e(t) = |z(t)|$.

Al calcular la PSD de $e(t)$ (restándole antes su valor medio, que si no aparece como una delta enorme en continua) la periodicidad de los impactos queda trasladada a baja frecuencia, junto con sus armónicos.

### Por qué una falla en pista interna tiene bandas laterales a $\pm f_r$

La pista interna gira con el eje, así que el defecto también gira. Una vez por vuelta pasa por la zona de máxima carga, donde el impacto es fuerte, y media vuelta después por la zona descargada, donde es débil. Entonces la amplitud de los impactos queda modulada a la frecuencia de giro:

$$e(t) \;\propto\; \underbrace{\left[1 + m\cos(2\pi f_r t)\right]}_{\text{modulación por zona de carga}}
\cdot \underbrace{\sum_i \delta(t - i/f_{\text{BPFI}})}_{\text{tren de impactos}}$$

Como un producto en el tiempo es una convolución en frecuencia, cada armónico $k\cdot f_{\text{BPFI}}$ aparece acompañado de bandas laterales en $k\cdot f_{\text{BPFI}} \pm f_r$. Esa es la firma de la falla en pista interna y lo que la diferencia de una falla en pista externa, que está fija respecto de la carga y no genera esas bandas.

In [ ]:
# ENV : cada canal demodulado en SU PROPIA banda optima (procedimiento automatico completo)
# ENVC: todos los canales demodulados en la MISMA banda, la del canal bajo estudio.
#       Esta ultima es la que permite comparar: pregunta si hay energia a f_BPFI en la
#       banda de resonancia de interes, con parametros identicos para todos.
ENV, ENVC = {}, {}
for clave, d in CH.items():
    banda = KURT[clave]["banda"]
    f_e, P_e, env, y_b = espectro_envolvente(d["x"], d["fs"], banda)
    ENV[clave] = {"f": f_e, "P": P_e, "env": env, "filtrada": y_b, "banda": banda}

    f_c, P_c, env_c, y_c = espectro_envolvente(d["x"], d["fs"], BANDA)
    ENVC[clave] = {"f": f_c, "P": P_c, "env": env_c, "filtrada": y_c, "banda": BANDA}

claves_maq = [k for k in CH if k != "RUIDO"]

# Pico dominante provisorio, para poder anotar los graficos que siguen. El paso 7 lo
# vuelve a calcular con el procedimiento formal y reporta la evidencia que lo sostiene.
_f, _P = ENVC["M2-C1"]["f"], ENVC["M2-C1"]["P"]
_m = (_f >= 2 * fr) & (_f <= 15 * fr)
f_pico_prov = float(_f[_m][np.argmax(_P[_m])])

print(f"Resolucion del espectro de envolvente: {ENV['M2-C1']['f'][1]:.3f} Hz "
      f"(fr / {fr/ENV['M2-C1']['f'][1]:.0f})")
print(f"Banda comun de comparacion: {BANDA[0]:.0f} - {BANDA[1]:.0f} Hz")
print(f"Pico dominante provisorio en M2-C1: {f_pico_prov:.2f} Hz "
      f"-> periodo entre impactos {1000/f_pico_prov:.2f} ms")

In [ ]:
# Efecto de la demodulacion, en el tiempo
d = CH["M2-C1"]
e = ENV["M2-C1"]
n0, n1 = 0, int(0.10 * FS_ACEL)       # 100 ms
tt = np.arange(n0, n1) / FS_ACEL * 1e3

fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
ax[0].plot(tt, d["x"][n0:n1], lw=0.5, color="#95a5a6", label="señal cruda")
ax[0].set(ylabel="[g]", title="M2-C1: señal cruda")
ax[0].legend(loc="upper right")

ax[1].plot(tt, e["filtrada"][n0:n1], lw=0.6, color=COLOR["falla"], alpha=0.8,
           label=f"filtrada {e['banda'][0]:.0f}-{e['banda'][1]:.0f} Hz")
ax[1].plot(tt, e["env"][n0:n1] + e["env"].mean(), lw=1.3, color="k",
           label="envolvente |z(t)|")
ax[1].set(xlabel="Tiempo [ms]", ylabel="[g]",
          title="Banda de resonancia y su envolvente")
ax[1].legend(loc="upper right")
fig.tight_layout()
plt.show()

*Figura 14. 100 ms de M2-C1: la señal cruda (arriba) y la misma señal filtrada en 1500–3000 Hz con su envolvente $|z(t)|$ (abajo). El filtrado y la transformada de Hilbert separan la envolvente de la portadora, y es esa curva, no la oscilación de kHz, la que analiza el espectro de envolvente.*

In [ ]:
# --- la envolvente en el TIEMPO, los cuatro canales en la misma banda y misma escala ---
n0, n1 = 0, int(0.10 * FS_ACEL)
tt = np.arange(n0, n1) / FS_ACEL * 1e3
Td_ms = 1000.0 / f_pico_prov          # periodo entre impactos esperado

fig, axes = plt.subplots(len(claves_maq), 1, figsize=(12, 1.9 * len(claves_maq)),
                         sharex=True, sharey=True)
for a, clave in zip(axes, claves_maq):
    e = ENVC[clave]
    a.plot(tt, e["filtrada"][n0:n1], lw=0.5, color="#bdc3c7")
    a.plot(tt, e["env"][n0:n1] + e["env"].mean(), lw=1.3, color=COLOR[CH[clave]["rol"]])
    for k in range(int(0.10 * 1000 / Td_ms) + 1):
        a.axvline(k * Td_ms, color="k", ls=":", lw=0.8, alpha=0.55)
    a.set_ylabel(f"{clave}\n[g]")
axes[0].set_title(f"Envolvente en el tiempo, banda {BANDA[0]:.0f}-{BANDA[1]:.0f} Hz. "
                  f"Punteado: multiplos de $T_d$ = {Td_ms:.2f} ms")
axes[-1].set_xlabel("Tiempo [ms]")
fig.tight_layout()
plt.show()

*Figura 15. Envolvente (línea gruesa) de los cuatro canales de máquina en la misma banda (1500–3000 Hz) y con la misma escala, durante 100 ms. Las líneas punteadas son múltiplos de $T_d$ = 5.77 ms. Solo en M2-C1 los picos de la envolvente caen regularmente sobre las punteadas; en los canales de referencia fluctúa sin periodicidad visible.*

Acá el fenómeno se ve directamente en el tiempo, antes de cualquier transformada. En M2-C1 la envolvente (línea gruesa) tiene **picos regulares** que caen sobre las líneas punteadas, separadas $T_d = 5.77$ ms. En los tres canales de referencia la envolvente sube y baja sin estructura, como se espera de una banda de ruido filtrado.

Esto es toda la física del método. Lo que sigue (el espectro de envolvente, la identificación del paso 7, el tiempo-frecuencia) sirve para medir esa periodicidad de forma objetiva y automática, pero el fenómeno ya se ve acá.

In [ ]:
# --- espectro de envolvente del canal con falla, anotado ---
f_e, P_e = ENV["M2-C1"]["f"], ENV["M2-C1"]["P"]

# pico dominante empirico en 2*fr .. 15*fr (ver paso 7)
msk = (f_e >= 2 * fr) & (f_e <= 15 * fr)
f_pico = float(f_e[msk][np.argmax(P_e[msk])])

fig, ax = plt.subplots(2, 1, figsize=(12.5, 7))

ax[0].plot(f_e, P_e, lw=0.8, color=COLOR["falla"])
marcar(ax[0], [k * f_pico for k in range(1, 5)],
       [f"{k}x pico" for k in range(1, 5)], color="k", ls="-", alpha=0.8, y_rel=0.97)
marcar(ax[0], [TEO["BPFI"]], ["BPFI teorico"], color="#8e44ad", ls="-.", y_rel=0.62)
marcar(ax[0], [TEO["BPFO"]], ["BPFO teorico"], color="#16a085", ls="-.", y_rel=0.40)
marcar(ax[0], [k * fr for k in range(1, 4)], [f"{k}x fr" for k in range(1, 4)],
       color="#7f8c8d", ls=":", y_rel=0.22)
ax[0].set(xlabel="Frecuencia [Hz]", ylabel=r"PSD envolvente [$g^2$/Hz]",
          xlim=(0, 800), yscale="log",
          ylim=(P_e[f_e > 1].max() * 1e-4, P_e[f_e > 1].max() * 4),
          title=f"M2-C1: espectro de envolvente (banda {ENV['M2-C1']['banda'][0]:.0f}"
                f"-{ENV['M2-C1']['banda'][1]:.0f} Hz)")

# zoom en el primer armonico, para ver las bandas laterales
ax[1].plot(f_e, P_e, lw=1.0, color=COLOR["falla"])
marcar(ax[1], [f_pico], [f"{f_pico:.1f} Hz"], color="k", ls="-")
marcar(ax[1], [f_pico - 2 * fr, f_pico - fr, f_pico + fr, f_pico + 2 * fr],
       [r"$-2f_r$", r"$-f_r$", r"$+f_r$", r"$+2f_r$"], color="#2980b9", ls="--")
ax[1].set(xlabel="Frecuencia [Hz]", ylabel=r"PSD envolvente [$g^2$/Hz]",
          xlim=(f_pico - 3.2 * fr, f_pico + 3.2 * fr),
          title="Detalle del pico dominante: bandas laterales espaciadas $f_r$")
fig.tight_layout()
plt.show()

print(f"Pico dominante: {f_pico:.2f} Hz = {f_pico/fr:.3f} x fr")

*Figura 16. Arriba: espectro de envolvente de M2-C1 (banda 1500–3000 Hz, escala logarítmica) con el pico dominante y sus armónicos (negro), el BPFI y el BPFO teóricos (rayas y puntos) y los primeros armónicos de $f_r$ (punteado). Abajo: zoom lineal alrededor del pico dominante con las bandas laterales a $\pm f_r$ y $\pm 2f_r$. La serie empieza en 173.2 Hz, corrida respecto del BPFI teórico de 180.4 Hz (los armónicos encontrados se imprimen en el paso 7), y abajo se ven las bandas laterales en 156.4 y 189.7 Hz.*

In [ ]:
# --- control negativo: los demas canales, en la MISMA banda y con los mismos parametros ---
# escala vertical COMUN a los cuatro paneles, para que la comparacion sea directa
_tope = max(ENVC[k]["P"][ENVC[k]["f"] > 1].max() for k in claves_maq)
YLIM = (_tope * 1e-4, _tope * 4)

fig, ax = plt.subplots(len(claves_maq), 1, figsize=(12, 2.3 * len(claves_maq)),
                       sharex=True, sharey=True)
for a, clave in zip(ax, claves_maq):
    e = ENVC[clave]
    a.plot(e["f"], e["P"], lw=0.8, color=COLOR[CH[clave]["rol"]])
    marcar(a, [k * f_pico for k in range(1, 4)],
           [f"{k}x" for k in range(1, 4)], color="k", ls="-", alpha=0.7)
    a.set(ylabel=f"{clave}\n" + r"[$g^2$/Hz]", yscale="log", xlim=(0, 700), ylim=YLIM)
    a.set_title(f"{clave} — {CH[clave]['desc']}", fontsize=9)
ax[-1].set_xlabel("Frecuencia [Hz]")
fig.suptitle(f"Control negativo: misma banda ({BANDA[0]:.0f}-{BANDA[1]:.0f} Hz) y "
             f"mismos parametros para todos los canales", y=1.005, fontsize=11)
fig.tight_layout()
plt.show()

print(f"Relacion pico/fondo a {f_pico:.1f} Hz, por canal:")
tol_c = max(0.03 * f_pico, 3 * (ENVC['M2-C1']['f'][1] - ENVC['M2-C1']['f'][0]))
for clave in claves_maq:
    _, _, s = pico_cerca(ENVC[clave]["f"], ENVC[clave]["P"], f_pico, tol_c)
    print(f"  {clave:<6}: {s:5.1f} dB")

print(f"\nCada canal en SU banda optima del paso 5: pico dominante de la envolvente entre 2 y 15 fr")
for clave in claves_maq:
    fe, Pe = ENV[clave]["f"], ENV[clave]["P"]
    mm = (fe >= 2 * fr) & (fe <= 15 * fr)
    fp = fe[mm][np.argmax(Pe[mm])]
    ent = round(fp / fr)
    cerca = min(COEF, key=lambda k: abs(fp / fr - COEF[k]))
    _, _, s = pico_cerca(fe, Pe, f_pico, tol_c)
    print(f"  {clave:<6} {ENV[clave]['banda'][0]:.0f}-{ENV[clave]['banda'][1]:.0f} Hz: "
          f"{fp:.2f} Hz = {fp / fr:.2f} x fr  (entero mas cercano {ent}: "
          f"{100 * abs(fp / fr - ent) / ent:.1f} %; frec. del rodamiento mas cercana {cerca}: "
          f"{100 * (fp / fr - COEF[cerca]) / COEF[cerca]:+.1f} %)  |  en {f_pico:.1f} Hz: {s:.1f} dB")

*Figura 17. Espectro de envolvente de los cuatro canales de máquina en la misma banda (1500–3000 Hz), con la misma escala vertical y los tres primeros armónicos de 173.2 Hz marcados. La serie aparece solo en M2-C1, 20.6 dB sobre el fondo local, contra 1.3 a 1.4 dB en los canales de referencia.*

La diferencia es clara: solo M2-C1 tiene una serie armónica en $173$ Hz, 20.6 dB sobre el fondo local, mientras que los canales de referencia quedan en 1.3 y 1.4 dB, es decir, al nivel del fondo. Los cuatro canales se procesaron con **exactamente los mismos parámetros** (misma banda, mismo filtro, misma demodulación, misma estimación espectral), así que el resultado no es un artefacto del procesamiento.

Si demodulamos cada canal en su propia banda óptima (segunda parte del print), ningún canal de referencia tiene energía en 173.2 Hz: quedan entre 1.0 y 1.6 dB sobre el fondo. M2-C2 y M1-C2 tienen su pico dominante en múltiplos enteros de $f_r$ ($2\times$ y $5\times$), o sea, componentes sincrónicas. M1-C1 lo tiene en 127.08 Hz $= 7.63\times f_r$, que no es entero y está a −6.7 % del BPFO teórico. No evaluamos su serie armónica ni sus bandas laterales, así que queda señalado pero no diagnosticado.

### Validación cruzada con el software de mantenimiento

El sistema de monitoreo también exporta su propia envolvente (`med2_env_wf.csv`, en gE, muestreada a 3 kHz). Su algoritmo es propietario y no sabemos qué banda ni qué método de demodulación usa, pero si nuestra cadena está bien, los dos espectros deberían coincidir en la frecuencia de los picos, aunque no necesariamente en las amplitudes.

Es una **verificación independiente** de todo el pipeline: carga, escalado, filtrado, Hilbert y estimación espectral.

In [ ]:
_, E, fse, _ = cargar_wf("med2_env_wf.csv", FS_LF, t_max=T_ANALISIS)
env_sw = preparar(E[:, 0])                       # canal 1, el del lado acople
f_sw, P_sw = signal.welch(env_sw, fse, nperseg=2 ** 13, noverlap=2 ** 12)

fig, ax = plt.subplots(figsize=(12, 4.2))
ax.semilogy(f_e, P_e / P_e[(f_e > 2 * fr) & (f_e < 15 * fr)].max(),
            lw=0.9, color=COLOR["falla"], label="envolvente propia (Hilbert)")
ax.semilogy(f_sw, P_sw / P_sw[(f_sw > 2 * fr) & (f_sw < 15 * fr)].max(),
            lw=0.9, color="#e67e22", alpha=0.85,
            label="envolvente del software de mantenimiento")
marcar(ax, [k * f_pico for k in range(1, 5)], [f"{k}x" for k in range(1, 5)],
       color="k", ls="-", alpha=0.7)
ax.set(xlabel="Frecuencia [Hz]", ylabel="PSD normalizada", xlim=(0, 800),
       ylim=(1e-4, 3),
       title="Validacion cruzada del analisis de envolvente (M2-C1)")
ax.legend()
fig.tight_layout()
plt.show()

m_sw = (f_sw >= 2 * fr) & (f_sw <= 15 * fr)
f_pico_sw = float(f_sw[m_sw][np.argmax(P_sw[m_sw])])
print(f"Pico dominante, envolvente propia   : {f_pico:.2f} Hz")
print(f"Pico dominante, envolvente del equipo: {f_pico_sw:.2f} Hz")
print(f"Discrepancia: {abs(f_pico-f_pico_sw):.2f} Hz "
      f"({100*abs(f_pico-f_pico_sw)/f_pico:.2f} %, resolucion del equipo "
      f"{f_sw[1]-f_sw[0]:.2f} Hz)")

*Figura 18. Espectro de nuestra envolvente de M2-C1 (rojo) y de la envolvente que exporta el software de mantenimiento (naranja), cada uno normalizado por su máximo entre $2f_r$ y $15f_r$; por eso el pico de $f_r$ puede pasar de 1. Las amplitudes no tienen por qué coincidir, pero las frecuencias sí, y coinciden: los dos dan el pico dominante en 173.22 Hz.*

### El cepstrum de la envolvente

El espectro de envolvente de M2-C1 tiene una estructura muy reconocible: picos repetidos cada 173.2 Hz. El cepstrum está pensado para detectar justamente eso, una periodicidad dentro del espectro. Es la transformada inversa del logaritmo del espectro:

$$c(\tau) = \mathcal{F}^{-1}\left\{\log P_{\text{env}}(f)\right\}$$

La variable $\tau$ se llama quefrencia y tiene unidades de tiempo. Si el espectro tiene una familia de armónicos separados $\Delta f$, su logaritmo se repite cada $\Delta f$ a lo largo del eje de frecuencia, y la transformada inversa junta esa repetición en un solo pico, el rahmónico, en $\tau = 1/\Delta f$. Para nuestra serie, $\tau = 1/173.2\ \text{Hz} = 5.77$ ms, el mismo $T_d$ entre impactos que vimos en la Figura 15.

El logaritmo hace dos cosas. Comprime el rango dinámico, así que los armónicos débiles pesan casi tanto como el fundamental. Y convierte productos en sumas: el espectro de un tren de impactos es el peine de armónicos multiplicado por la forma de la banda, y en el logaritmo esa forma queda sumada, en las quefrencias bajas, separada de la periodicidad. El costo es que el logaritmo también realza los valles del ruido, que pasan a pesar tanto como los picos.

La pregunta es si **juntar todos los armónicos en un solo pico** mejora la relación pico/fondo de 20.6 dB de la Figura 17. Para responderla calculamos el cepstrum de los cuatro canales con la misma banda y los mismos parámetros. Los dB no sirven para comparar, porque el cepstrum ya está en escala logarítmica y toma valores negativos, así que usamos la misma vara en los dos dominios: la altura del pico sobre la mediana de su entorno, medida en desvíos robustos (MAD).

In [ ]:
# Cepstrum de potencia de la envolvente: c(tau) = IFFT{ log P_env(f) }.
# La envolvente de una banda de 1500 Hz de ancho no tiene contenido por encima de 1500 Hz,
# asi que el logaritmo se toma solo hasta F_CEP. Si se usara todo el eje hasta fs/2, la
# mayor parte del logaritmo seria ruido fuera de banda y taparia la periodicidad.
# La resolucion en quefrencia queda en 1 / (2 F_CEP) = 0.33 ms.
F_CEP = BANDA[1] - BANDA[0]            # Hz


def cepstrum_env(env, nperseg, fs=FS_ACEL, f_max=F_CEP):
    # Devuelve (quefrencia [s], cepstrum real) de una envolvente.
    f, P = signal.welch(env - env.mean(), fs, nperseg=nperseg, noverlap=nperseg // 2)
    m = f <= f_max
    c = np.fft.irfft(np.log(P[m]))
    return np.arange(len(c)) / (2 * f[m][-1]), c


def prominencia(x, y, x0, tol_rel=0.03):
    # (posicion, altura) del maximo en x0 +- tol, con la altura medida sobre el fondo
    # local en desvios robustos: mediana y MAD de un entorno de +-50 % de x0, sin el pico.
    tol = max(tol_rel * x0, 1.5 * (x[1] - x[0]))
    m = np.abs(x - x0) <= tol
    ent = (np.abs(x - x0) <= 0.5 * x0) & ~m
    med = np.median(y[ent])
    mad = 1.4826 * np.median(np.abs(y[ent] - med))
    j = np.argmax(y[m])
    return x[m][j], (y[m][j] - med) / mad


T_d = 1.0 / f_pico                     # periodo entre impactos: el rahmonico esperado
CEP_REG = {k: cepstrum_env(ENVC[k]["env"], nperseg=2 ** 15) for k in claves_maq}

_q = CEP_REG["M2-C1"][0]
_vis = (_q >= 3e-3) & (_q <= 70e-3)    # las quefrencias bajas son la forma del espectro
_tope = max(np.abs(CEP_REG[k][1][_vis]).max() for k in claves_maq)

fig, ax = plt.subplots(len(claves_maq), 1, figsize=(12, 2.3 * len(claves_maq)),
                       sharex=True, sharey=True)
for a, clave in zip(ax, claves_maq):
    q, c = CEP_REG[clave]
    a.plot(q * 1e3, c, lw=0.8, color=COLOR[CH[clave]["rol"]])
    marcar(a, [k * T_d * 1e3 for k in range(1, 4)],
           [f"{k}$T_d$" if k > 1 else "$T_d$" for k in range(1, 4)],
           color="k", ls="-", alpha=0.7)
    marcar(a, [1e3 / TEO["BPFI"]], ["BPFI teorico"], color="#8e44ad", ls="-.", y_rel=0.55)
    marcar(a, [1e3 / fr], ["$1/f_r$"], color="#7f8c8d", ls=":")
    a.set(ylabel=clave, xlim=(3, 70), ylim=(-1.1 * _tope, 1.1 * _tope))
    a.set_title(f"{clave} — {CH[clave]['desc']}", fontsize=9)
ax[-1].set_xlabel("Quefrencia [ms]")
fig.suptitle(f"Cepstrum de la envolvente: misma banda ({BANDA[0]:.0f}-{BANDA[1]:.0f} Hz) "
             f"y mismos parametros para todos los canales", y=1.005, fontsize=11)
fig.tight_layout()
plt.show()

# La misma medida de prominencia, aplicada al espectro de envolvente en dB y al cepstrum,
# para comparar los dos dominios con la misma vara.
print(f"Prominencia sobre el fondo local, en desvios robustos (T_d = {T_d*1e3:.2f} ms):")
print(f"{'canal':<7}{'espectro @ f_pico':>19}{'cepstrum @ T_d':>16}{'cepstrum @ 1/fr':>17}"
      f"{'pico/fondo [dB]':>17}")
for clave in claves_maq:
    fe, Pe = ENVC[clave]["f"], ENVC[clave]["P"]
    _, z_esp = prominencia(fe, 10 * np.log10(Pe), f_pico)
    q, c = CEP_REG[clave]
    q_p, z_cep = prominencia(q, c, T_d)
    _, z_fr = prominencia(q, c, 1 / fr)
    _, _, s = pico_cerca(fe, Pe, f_pico, max(0.03 * f_pico, 3 * (fe[1] - fe[0])))
    print(f"{clave:<7}{z_esp:>19.1f}{z_cep:>16.1f}{z_fr:>17.1f}{s:>17.1f}")
print(f"Rahmonico de M2-C1 en {prominencia(*CEP_REG['M2-C1'], T_d)[0]*1e3:.2f} ms "
      f"(resolucion en quefrencia {1e3 / (2 * F_CEP):.2f} ms)")

*Figura 19. Cepstrum de la envolvente de los cuatro canales de máquina (banda 1500–3000 Hz, logaritmo tomado hasta 1500 Hz, misma escala vertical). Las líneas negras marcan $T_d$ = 5.77 ms y sus múltiplos, la de rayas y puntos el BPFI teórico (5.54 ms) y la punteada $1/f_r$ = 60 ms. No se muestran las quefrencias menores a 3 ms, que corresponden a la forma general del espectro.*

El rahmónico aparece solo en M2-C1, en 5.67 ms. Los canales de referencia no tienen nada en esa quefrencia (entre −0.4 y 1.0 desvíos), así que el cepstrum confirma la periodicidad por un camino distinto del espectro. Con 0.33 ms de resolución en quefrencia no puede distinguir $T_d$ (5.77 ms) del BPFI teórico (5.54 ms); esa discriminación la sigue haciendo el espectro de envolvente, que resuelve 0.37 Hz.

Lo que no hace es mejorar la relación pico/fondo. Con la misma medida, el pico del espectro de envolvente de M2-C1 está 11.9 desvíos sobre su fondo y el rahmónico solo 3.8. El cepstrum rinde cuando hay muchos armónicos de amplitud parecida repartidos en todo el eje, porque cada uno suma al mismo pico. Acá la serie decae rápido: de 20.6 dB en el primer armónico a 6.0 dB en el quinto (866 Hz), y del sexto en adelante queda a menos de 3 dB del fondo. Entre 900 y 1500 Hz el espectro es casi todo ruido. El logaritmo le da a cada bin de ruido el mismo peso que a un armónico, así que esa parte del eje aporta varianza y nada de señal.

En M2-C1 también hay estructura cerca de $1/f_r$ = 60 ms (4.5 desvíos, contra 1.7 a 2.6 en las referencias). Es el rastro cepstral de las bandas laterales a $\pm f_r$ de la Figura 16, coherente con un defecto que gira con el eje.

In [ ]:
# C_BPFI: la prominencia del rahmonico en T_d, ventana a ventana de 1 s.
# nperseg = 2048 a 24 kHz da Delta f = 11.7 Hz: alcanza para resolver armonicos separados
# 173 Hz y promedia ~22 segmentos por ventana, lo que baja la varianza del logaritmo.
NPERSEG_CEP = 2048


def c_bpfi(env, f_obj, nperseg=NPERSEG_CEP):
    q, c = cepstrum_env(env, nperseg)
    return prominencia(q, c, 1.0 / f_obj)[1]


CEP = {k: np.array([c_bpfi(v, f_pico) for v in segmentar(ENVC[k]["env"], FS_ACEL, T_VENTANA)])
       for k in CH}

print(f"C_BPFI por ventana de {T_VENTANA:.0f} s (desvios robustos sobre el fondo cepstral):")
for clave in CH:
    v = CEP[clave]
    print(f"  {clave:<6}: mediana {np.median(v):5.2f}   min {v.min():5.2f}   max {v.max():5.2f}")

Llevamos la misma medida a ventanas de 1 s para tener un indicador comparable con $R_{\text{BPFI}}$, al que llamamos $C_{\text{BPFI}}$. La mediana en M2-C1 es 3.70 desvíos, contra 0.38 a 0.73 en las referencias y 0.79 en el ruido. Separa, pero las referencias tienen ventanas sueltas con valores altos: M2-C2 llega a 9.02 en una. En el paso 9 lo ponemos en las curvas ROC y DET junto a los otros indicadores, que es la forma justa de decidir si suma algo.

---

# 7. Identificación empírica de la falla

Este es el paso central del trabajo: **qué elemento del rodamiento está fallado**, respondido a partir de la señal y sin suponer la geometría interna.

Nos apoyamos en tres evidencias:

1. La relación $f_{\text{pico}}/f_r$. Las frecuencias de falla son múltiplos no enteros de la velocidad de giro, fijados por la cinemática del rodamiento. Un múltiplo no entero descarta de entrada las fuentes sincrónicas (desbalanceo, paso de álabes, engranajes).
2. La serie armónica. Un tren de impactos periódico genera armónicos $2\times$, $3\times$, $4\times$ del período fundamental; una resonancia excitada al azar, no.
3. Las bandas laterales a $\pm f_r$. Como vimos en el paso 6, indican que la fuente de los impactos gira con el eje, o sea pista interna y no externa.

Recién después comparamos con los coeficientes del reporte SKF, y esa comparación sirve para confirmar la identificación, no para producirla.

Sobre la ventana de búsqueda: buscamos el pico dominante entre $2f_r$ y $15 f_r$. El límite inferior deja afuera la componente $1\times$ y la continua, y el superior cubre de sobra el BPFI de este rodamiento ($10.83\,f_r$). Con una ventana de $2$–$10\,f_r$, que serviría para un rodamiento de bolas de 8–12 elementos, el pico quedaría afuera.

In [ ]:
def identificar_falla(f, P, fr, f_lo_mult=2.0, f_hi_mult=15.0, n_arm=4, tol_mult=0.02):
    # Identificacion empirica: pico dominante, armonicos y bandas laterales a +-fr.
    m = (f >= f_lo_mult * fr) & (f <= f_hi_mult * fr)
    f_pico = float(f[m][np.argmax(P[m])])
    tol = max(tol_mult * f_pico, 3 * (f[1] - f[0]))

    armonicos, laterales = [], []
    for k in range(1, n_arm + 1):
        fk, ak, snr = pico_cerca(f, P, k * f_pico, tol)
        armonicos.append({"k": k, "f_teorico": k * f_pico, "f_hallado": fk,
                          "SNR_dB": snr})
        for signo in (-1, +1):
            fl, al, snrl = pico_cerca(f, P, k * f_pico + signo * fr, tol)
            laterales.append({"k": k, "lado": "+fr" if signo > 0 else "-fr",
                              "f_teorico": k * f_pico + signo * fr,
                              "f_hallado": fl, "SNR_dB": snrl})
    return f_pico, pd.DataFrame(armonicos), pd.DataFrame(laterales)


f_pico, ARM, LAT = identificar_falla(f_e, P_e, fr)

print("=" * 66)
print(f"  PICO DOMINANTE DEL ESPECTRO DE ENVOLVENTE (M2-C1)")
print("=" * 66)
print(f"  f_pico = {f_pico:.2f} Hz")
print(f"  f_pico / fr = {f_pico/fr:.3f}   (fr = {fr:.3f} Hz = {fr*60:.0f} rpm)")
print(f"  Multiplo entero mas cercano: {round(f_pico/fr)}  -> desvio "
      f"{100*abs(f_pico/fr - round(f_pico/fr))/round(f_pico/fr):.1f} % "
      f"(NO es una componente sincronica)")
print("=" * 66)

print("\nSerie armonica:")
display(ARM.style.format({"f_teorico": "{:.2f}", "f_hallado": "{:.2f}",
                          "SNR_dB": "{:.1f}"}))
print("Bandas laterales a +- fr:")
display(LAT.style.format({"f_teorico": "{:.2f}", "f_hallado": "{:.2f}",
                          "SNR_dB": "{:.1f}"}))

In [ ]:
# --- contraste contra los coeficientes del fabricante ---
filas = []
for nombre, coef in COEF.items():
    f_teo = coef * fr
    filas.append({
        "frecuencia": nombre,
        "coef [x fr]": coef,
        "f teorica [Hz]": f_teo,
        "error [Hz]": f_pico - f_teo,
        "error [%]": 100 * (f_pico - f_teo) / f_teo,
    })
comp = pd.DataFrame(filas).set_index("frecuencia")
comp["|error| [%]"] = comp["error [%]"].abs()
comp = comp.sort_values("|error| [%]")
display(comp.style.format({"coef [x fr]": "{:.4f}", "f teorica [Hz]": "{:.2f}",
                           "error [Hz]": "{:+.2f}", "error [%]": "{:+.2f}",
                           "|error| [%]": "{:.2f}"}))

mejor_match = comp.index[0]
err = comp.loc[mejor_match, "error [%]"]
print(f"\nMejor coincidencia: {mejor_match}  ({err:+.2f} %)")
print(f"Siguiente candidato: {comp.index[1]}  ({comp.loc[comp.index[1], 'error [%]']:+.2f} %)")

n_lat = int((LAT["SNR_dB"] > 3).sum())
n_arm = int((ARM["SNR_dB"] > 3).sum())
print(f"\nEvidencia estructural: {n_arm}/{len(ARM)} armonicos y "
      f"{n_lat}/{len(LAT)} bandas laterales por encima de 3 dB sobre el fondo local.")

### Diagnóstico

El pico dominante está en $10.4\times f_r$, un múltiplo claramente no entero de la velocidad de giro, con cuatro armónicos y bandas laterales a $\pm f_r$: 7 de las 8 superan los 3 dB sobre el fondo local (la que falta, $3\times - f_r$, queda en 2.8 dB). El candidato teórico más cercano es el BPFI, con un error de $-3.97\%$.

**Diagnóstico: falla localizada en la pista interna del rodamiento del lado acople del ventilador 2 (canal M2-C1).**

### Sobre el desvío del 4 %

El BPFI teórico del reporte SKF es $10.826\,f_r$ y el medido es $10.396\,f_r$. La diferencia no invalida el diagnóstico y tiene causas físicas conocidas.

La primera es el deslizamiento (*slip*). Las fórmulas cinemáticas suponen rodadura pura, pero en la práctica los elementos rodantes patinan, sobre todo con poca carga radial o con juego interno holgado (este rodamiento es C3, de juego ampliado). El deslizamiento baja sistemáticamente la frecuencia real de paso, típicamente entre 1 % y 2 % según la literatura (Randall y Antoni, 2011), y puede ser más en rodamientos de rodillos a rótula grandes.

También influye el ángulo de contacto real. En un rodamiento a rótula, $\alpha$ depende de la relación entre carga axial y radial y del desalineamiento del soporte, y las fórmulas son muy sensibles a $\alpha$; los coeficientes del fabricante corresponden a una condición nominal que rara vez es la de operación. Por último, la carga se puede redistribuir entre las dos hileras del rodamiento.

Lo importante es que la identificación no dependió del valor teórico. Aun sin el reporte SKF, la combinación de múltiplo no entero de $f_r$, armónicos y bandas laterales a $\pm f_r$ ya indica una falla en un elemento que gira con el eje, y $10.4\times$ solo puede ser la pista interna: con $Z = 19$, el BPFI queda por encima de $Z/2 = 9.5\times f_r$ y el BPFO por debajo. El reporte del fabricante confirma la lectura, no la produce.

In [ ]:
# --- preparado para revalidar con la geometria real, si se consigue ---
Z_REAL = Z          # <-- reemplazar por el Z de fabrica si se obtiene la geometria
GAMMA_REAL = gamma  # <-- (Dw/Dpw)*cos(alpha)

bpfi_geo = (Z_REAL / 2) * fr * (1 + GAMMA_REAL)
bpfo_geo = (Z_REAL / 2) * fr * (1 - GAMMA_REAL)
ftf_geo  = 0.5 * fr * (1 - GAMMA_REAL)

print("Recalculo a partir de la geometria (Z y Dw/Dpw*cos(alpha)):")
print(f"  BPFI = {bpfi_geo:7.2f} Hz   BPFO = {bpfo_geo:7.2f} Hz   FTF = {ftf_geo:6.2f} Hz")
print(f"  Consistencia BPFO + BPFI = Z*fr : "
      f"{bpfi_geo + bpfo_geo:.2f} vs {Z_REAL*fr:.2f} Hz")
print(f"  Consistencia BPFO = Z*FTF       : "
      f"{bpfo_geo:.2f} vs {Z_REAL*ftf_geo:.2f} Hz")
print(f"\n  Deslizamiento implicito = {100*(1 - f_pico/bpfi_geo):+.2f} %")

In [ ]:
# --- sensibilidad a la duracion del registro analizado ---
print("Estabilidad del pico dominante frente a T_ANALISIS:")
print(f"{'T [s]':>7} {'N':>10} {'f_pico [Hz]':>13} {'x fr':>8}")
for T in [5.0, 10.0, 20.0, 40.0, T_ANALISIS]:
    n = int(FS_ACEL * T)
    if n > len(x_falla):
        continue
    nps = min(NPERSEG_ENV, 1 << int(np.floor(np.log2(n / 4))))
    fa, Pa, _, _ = espectro_envolvente(x_falla[:n], FS_ACEL, BANDA, nperseg=nps)
    ma = (fa >= 2 * fr) & (fa <= 15 * fr)
    fp = fa[ma][np.argmax(Pa[ma])]
    print(f"{T:7.0f} {n:10,d} {fp:13.2f} {fp/fr:8.3f}")
print(f"\nResolucion espectral a T = {T_ANALISIS:.0f} s: {FS_ACEL/NPERSEG_ENV:.3f} Hz")

---

# 8. Análisis tiempo-frecuencia

El espectro de envolvente muestra que hay una periodicidad a $f_{\text{BPFI}}$, pero es un promedio sobre todo el registro: un tren de impactos permanente y una única ráfaga muy energética pueden dar picos parecidos. Falta ver **si los impactos son persistentes**, que es lo que distingue un defecto establecido de un evento espurio (un golpe, una interferencia eléctrica, alguien que pasa cerca).

### STFT

$$X(\tau, f) = \sum_{n} x[n]\,w[n-\tau]\,e^{-j2\pi f n/f_s}$$

La ventana $w$ fija la resolución en los dos ejes a la vez, con el límite de incertidumbre $\Delta t \cdot \Delta f \geq 1/4\pi$. Para ver impactos separados $T_d = 1/f_{\text{BPFI}} = 5.77$ ms, la ventana tiene que ser bastante más corta que eso.

### Transformada wavelet continua (Morlet)

$$W(s,\tau) = \frac{1}{\sqrt{s}}\int x(t)\,\psi^{*}\!\left(\frac{t-\tau}{s}\right)dt,
\qquad
\psi(t) = \pi^{-1/4} e^{j\omega_0 t} e^{-t^2/2}$$

La wavelet de Morlet es una sinusoide compleja modulada por una gaussiana. Al escalarla, la resolución temporal pasa a depender de la frecuencia: ventanas largas en baja frecuencia (buena resolución en frecuencia) y cortas en alta (buena resolución temporal), que es lo que conviene para transitorios de alta frecuencia como los impactos de rodamiento.

Usamos `pywt.cwt` con la wavelet `cmor` (Morlet compleja). PyWavelets la define por su ancho de banda $B$ y su frecuencia central $f_c$ (notación `cmor{B}-{fc}`) y trabaja con escalas en vez de frecuencias, así que hay que convertir:

$$s = \frac{f_c \cdot f_s}{f}$$

con $f_c = \omega_0/2\pi$. La función de abajo hace esa conversión para poder pedir directamente las frecuencias que queremos.

In [ ]:
import pywt
print("PyWavelets", pywt.__version__)


def cwt_morlet(x, fs, freqs, w0=6.0, ancho_banda=1.0):
    # CWT con wavelet de Morlet compleja, pidiendo directamente una grilla de frecuencias.
    # PyWavelets trabaja con escalas: s = f_c * fs / f, con f_c la frecuencia central
    # normalizada de la wavelet.
    nombre = f"cmor{ancho_banda}-{w0/(2*np.pi):.4f}"
    escalas = pywt.central_frequency(nombre) * fs / np.asarray(freqs, float)
    coef, _ = pywt.cwt(x, escalas, nombre, sampling_period=1 / fs)
    return coef

In [ ]:
T_TF = 0.12                                   # s de señal para el analisis tiempo-frecuencia
n_tf = int(T_TF * FS_ACEL)
x_tf = CH["M2-C1"]["x"][:n_tf]
y_tf = ENV["M2-C1"]["filtrada"][:n_tf]        # señal filtrada en la banda del kurtograma
e_tf = ENV["M2-C1"]["env"][:n_tf] + ENV["M2-C1"]["env"].mean()
tt = np.arange(n_tf) / FS_ACEL * 1e3
Td = 1000.0 / f_pico                          # periodo entre impactos, en ms

fig, ax = plt.subplots(3, 1, figsize=(12.5, 8), sharex=True,
                       gridspec_kw={"height_ratios": [1, 1.5, 1.5]})

# --- señal filtrada + envolvente + marcas del periodo esperado ---
ax[0].plot(tt, y_tf, lw=0.6, color="#95a5a6")
ax[0].plot(tt, e_tf, lw=1.2, color=COLOR["falla"])
for k in range(int(T_TF * 1000 / Td) + 1):
    ax[0].axvline(k * Td, color="k", ls=":", lw=0.8, alpha=0.7)
ax[0].set(ylabel="[g]",
          title=f"M2-C1 filtrada en {BANDA[0]:.0f}-{BANDA[1]:.0f} Hz. "
                f"Lineas punteadas: multiplos de $T_d$ = {Td:.2f} ms = 1/{f_pico:.1f} Hz")

# --- STFT ---
DIN = 35.0            # rango dinamico del mapa de color, en dB
nper = 256
# OJO con los nombres: la salida de stft se llama Zxx y no Z, porque Z ya es el numero
# de elementos rodantes del rodamiento (paso 0) y lo usa la celda de geometria del paso 7.
f_stft, t_stft, Zxx = signal.stft(x_tf, FS_ACEL, nperseg=nper, noverlap=nper - 32)
S_db = 20 * np.log10(np.abs(Zxx) + 1e-12)
ax[1].pcolormesh(t_stft * 1e3, f_stft, S_db, shading="gouraud", cmap="magma",
                 vmin=S_db.max() - DIN, vmax=S_db.max())
for k in range(int(T_TF * 1000 / Td) + 1):
    ax[1].axvline(k * Td, color="w", ls=":", lw=0.8, alpha=0.6)
ax[1].axhline(BANDA[0], color="c", ls="--", lw=1)
ax[1].axhline(BANDA[1], color="c", ls="--", lw=1)
ax[1].set(ylabel="Frecuencia [Hz]",
          title=f"STFT (ventana de {nper} muestras = {1000*nper/FS_ACEL:.1f} ms). "
                f"En cian, la banda del kurtograma")
ax[1].grid(False)

# --- CWT ---
frecs = np.geomspace(200, 11000, 100)
W = cwt_morlet(x_tf, FS_ACEL, frecs)
Wd = 20 * np.log10(np.abs(W) + 1e-12)
ax[2].pcolormesh(tt, frecs, Wd, shading="gouraud", cmap="magma",
                 vmin=Wd.max() - DIN, vmax=Wd.max())
for k in range(int(T_TF * 1000 / Td) + 1):
    ax[2].axvline(k * Td, color="w", ls=":", lw=0.8, alpha=0.6)
ax[2].set(xlabel="Tiempo [ms]", ylabel="Frecuencia [Hz]", yscale="log",
          title="CWT con wavelet de Morlet ($\\omega_0$ = 6)")
ax[2].grid(False)
fig.tight_layout()
plt.show()

*Figura 20. 120 ms de M2-C1. Arriba, la señal filtrada en 1500–3000 Hz y su envolvente; al medio, la STFT (ventana de 256 muestras = 10.7 ms) con la banda del kurtograma en cian; abajo, la CWT de Morlet con eje de frecuencia logarítmico. Las líneas punteadas son múltiplos de $T_d$ = 5.77 ms. La energía de alta frecuencia llega en estructuras verticales (impactos) alineadas con las punteadas, no como una franja horizontal continua.*

Los máximos de energía de alta frecuencia se alinean con las líneas punteadas, que marcan múltiplos del período $T_d = 1/f_{\text{BPFI}}$ calculado por separado a partir del espectro de envolvente. La CWT muestra cada impacto como una estructura vertical, concentrada en el tiempo y extendida en frecuencia, que es la firma de un transitorio de banda ancha. Una componente estacionaria se vería como una franja horizontal continua.

La alineación no es perfecta en todos los impactos. Eso es el **deslizamiento** de los rodillos ($\tau_i$ en el modelo del paso 6), que hace que el tren de impactos sea cuasiperiódico y no estrictamente periódico. Por lo mismo, el espectro de envolvente tiene picos de ancho finito en lugar de deltas, y un análisis de Fourier puro pierde parte de la energía del fenómeno.

Un detalle: en el espectrograma la banda más brillante está cerca de los 3.9 kHz del *hump* de la Figura 9, mientras que el kurtograma eligió 1500–3000 Hz. No es una contradicción, son dos criterios distintos. El mapa tiempo-frecuencia muestra dónde hay más energía, y la kurtosis espectral busca dónde hay más contraste entre los impactos y el fondo. Una banda puede ser la más energética y a la vez la peor para demodular si esa energía es estacionaria. Elegir la banda por el brillo del espectrograma, que es lo que haría el ojo, habría dado otra banda, con menor kurtosis espectral (Figura 13a).

In [ ]:
# Persistencia: relacion pico/fondo a f_BPFI, ventana a ventana, en todo el registro.
# Se mide como RELACION y no como energia absoluta: la energia absoluta depende del nivel
# global de vibracion del canal y no dice nada por si sola (ver paso 9).
def razon_bpfi(env, fs, f_obj, n_arm=3, tol_rel=0.03):
    nps = 1 << int(np.floor(np.log2(len(env) / 4)))
    f, P = signal.welch(env - env.mean(), fs, nperseg=nps, noverlap=nps // 2)
    tol = max(tol_rel * f_obj, 3 * (f[1] - f[0]))
    num = den = 0.0
    for k in range(1, n_arm + 1):
        m = np.abs(f - k * f_obj) <= tol
        mb = np.abs(f - k * f_obj) <= 10 * tol
        if m.any() and mb.any():
            num += np.trapezoid(P[m], f[m])
            den += np.median(P[mb]) * (f[m][-1] - f[m][0])
    return 10 * np.log10(num / den) if den > 0 else np.nan


V = segmentar(ENVC["M2-C1"]["env"], FS_ACEL, t_seg=T_VENTANA)
razon = np.array([razon_bpfi(v, FS_ACEL, f_pico) for v in V])

fig, ax = plt.subplots(figsize=(11.5, 3))
ax.plot(np.arange(len(razon)) * T_VENTANA, razon, "o-", ms=3, lw=0.9,
        color=COLOR["falla"])
ax.axhline(3.0, color="k", ls="--", lw=1)
ax.annotate("3 dB", xy=(0.99, 3.0), xycoords=("axes fraction", "data"),
            ha="right", va="bottom", fontsize=8)
ax.set(xlabel="Tiempo [s]", ylabel=r"$R_{BPFI}$ [dB]",
       title="Persistencia de la falla: relacion pico/fondo a $f_{BPFI}$ "
             "ventana a ventana")
fig.tight_layout()
plt.show()

print(f"R_BPFI por ventana: mediana {np.median(razon):.2f} dB, "
      f"min {razon.min():.2f} dB, max {razon.max():.2f} dB")
print(f"Ventanas con R_BPFI > 3 dB: {100*np.mean(razon > 3):.0f} % de "
      f"{len(razon)} ventanas de {T_VENTANA:.0f} s")

*Figura 21. Relación pico/fondo de la envolvente de M2-C1 en 173.2 Hz y sus armónicos, ventana a ventana de 1 s; la línea a trazos marca 3 dB. Nunca baja a cero (el mínimo es 1.03 dB), pero fluctúa: supera los 3 dB en el 62 % de las ventanas.*

$R_{\text{BPFI}}$ es positivo en todas las ventanas (el mínimo es 1.03 dB) y supera los 3 dB en el 62 % de ellas. La firma no desaparece en ningún tramo del registro, aunque su intensidad fluctúa, así que se trata de **un defecto establecido** y no de un evento transitorio.

---

# 9. Indicadores de salud y umbral de detección

Para que esto sirva como prototipo de mantenimiento predictivo hay que reducir el análisis a un vector de características por ventana, que se pueda comparar entre canales y entre mediciones sucesivas.

Usamos tres indicadores que se complementan:

| Indicador | Qué mide | Para qué está |
|---|---|---|
| $\text{Kurt}_b$ | impulsividad de la señal filtrada en la banda de resonancia | genérico: detecta la falla sin saber nada del rodamiento |
| $R_{\text{BPFI}}$ | relación pico/fondo de la envolvente en $f_{\text{BPFI}}$ y armónicos, en dB | específico de esta falla, muy selectivo |
| $\text{SNR}_{\text{banda}}$ | energía en la banda de resonancia respecto del piso de ruido del paso 2 | verifica que la medición tenga rango dinámico útil |

$R_{\text{BPFI}}$ es una relación y no una energía por una razón. La primera versión del indicador usaba la energía absoluta de la envolvente en $f_{\text{BPFI}}$ y no separaba nada: el canal sano M2-C2, que vibra el doble, tiene más energía absoluta en esa frecuencia que el canal con falla, simplemente porque todo su espectro de envolvente está más alto. Medido sobre los datos, el índice de Fisher de la energía absoluta es $J = 0.22$, frente a $J = 5.25$ de la relación pico/fondo (los dos se imprimen en la celda de separabilidad). Esto vale en general: **un indicador espectral de falla tiene que estar normalizado por el fondo local**, si no mide el nivel de vibración y no la falla.

Los tres se calculan con los mismos parámetros para todos los canales (misma banda de demodulación, misma frecuencia objetivo), porque si no la comparación no significa nada. En un sistema real esos parámetros se fijan una vez a partir del rodamiento instalado y se aplican en cada ronda de medición.

### El umbral

Tomamos como referencia las ventanas de los canales sanos y definimos

$$\text{umbral} \;=\; \mu_{\text{ref}} \;+\; k\,\sigma_{\text{ref}}, \qquad k = 3$$

Si los indicadores fueran aproximadamente normales, $k=3$ correspondería a una tasa de falsa alarma de $\sim 0.13\,\%$ por ventana. Hay dos salvedades. La referencia son pocas ventanas de dos equipos, no una población estadística real, así que el umbral es un prototipo metodológico y no una calibración industrial. Y los indicadores no son exactamente gaussianos (la kurtosis muestral tiene distribución asimétrica), así que la tasa nominal es solo orientativa.

Un sistema en funcionamiento usaría el histórico del propio equipo en estado sano, acumulado durante meses, y umbrales por percentiles en vez de por $\sigma$.

In [ ]:
FEATURES = ["Kurt_banda", "R_BPFI_dB", "SNR_banda_dB"]
FEAT_FALLA = ["Kurt_banda", "R_BPFI_dB"]      # los que deciden falla


def features_ventana(x, fs, banda, f_obj, psd_ruido, f_ruido, n_arm=3, tol_rel=0.03):
    # Vector de caracteristicas de una ventana. Usa SIEMPRE los mismos parametros
    # (banda y f_obj) para todos los canales, para que sean comparables.
    lo = max(banda[0], 1.0)
    hi = min(banda[1], 0.99 * fs / 2)
    sos = signal.butter(6, [lo, hi], btype="bandpass", fs=fs, output="sos")
    y = signal.sosfiltfilt(sos, x)
    env = np.abs(signal.hilbert(y))

    # potencia en banda vs. potencia de ruido en la misma banda
    f_p, P_p = signal.welch(x, fs, nperseg=min(2 ** 13, len(x)))
    mb = (f_p >= lo) & (f_p <= hi)
    p_sig = np.trapezoid(P_p[mb], f_p[mb])
    p_rui = np.trapezoid(np.interp(f_p[mb], f_ruido, psd_ruido), f_p[mb])

    return {
        "Kurt_banda": stats.kurtosis(y, bias=False),
        "R_BPFI_dB": razon_bpfi(env, fs, f_obj, n_arm=n_arm, tol_rel=tol_rel),
        "SNR_banda_dB": 10 * np.log10(p_sig / p_rui) if p_rui > 0 else np.nan,
    }


FEAT = {}
for clave, d in CH.items():
    V = segmentar(d["x"], d["fs"], t_seg=T_VENTANA)
    FEAT[clave] = pd.DataFrame(
        [features_ventana(v, d["fs"], BANDA, f_pico, PSD_RUIDO, F_RUIDO) for v in V]
    )
    print(f"{clave}: {len(FEAT[clave])} ventanas de {T_VENTANA:.0f} s")

In [ ]:
CANALES_REF = ["M2-C2", "M1-C1", "M1-C2"]     # poblacion sana de referencia
ref = pd.concat([FEAT[k] for k in CANALES_REF], ignore_index=True)

UMBRAL = {}
for col in FEAT_FALLA:
    mu, sd = ref[col].mean(), ref[col].std()
    UMBRAL[col] = mu + K_SIGMA * sd
    print(f"{col:<14}: mu_ref = {mu:+.4g}, sigma_ref = {sd:.4g} "
          f"-> umbral = {UMBRAL[col]:.4g}")
UMBRAL["SNR_banda_dB"] = 6.0                  # minimo de rango dinamico util
print(f"{'SNR_banda_dB':<14}: umbral fijo = {UMBRAL['SNR_banda_dB']:.1f} dB "
      f"(criterio de validez de la medicion, no de falla)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.8))
for ax, col in zip(axes, FEATURES):
    for clave in CH:
        ax.hist(FEAT[clave][col], bins=18, alpha=0.55, label=clave,
                color=COLOR[CH[clave]["rol"]])
    ax.axvline(UMBRAL[col], color="k", ls="--", lw=1.4)
    ax.annotate("umbral", xy=(UMBRAL[col], 0.93), xycoords=("data", "axes fraction"),
                rotation=90, ha="right", va="top", fontsize=8)
    ax.set(xlabel=col, ylabel="ventanas")
axes[0].legend(fontsize=8)
fig.suptitle(f"Distribucion de los indicadores por ventana de {T_VENTANA:.0f} s "
             f"(umbral = $\\mu_{{ref}}$ + {K_SIGMA:.0f}$\\sigma_{{ref}}$)", y=1.03)
fig.tight_layout()
plt.show()

# --- espacio de features: los dos indicadores de falla, juntos ---
fig, ax = plt.subplots(figsize=(6.5, 5))
for clave in CH:
    ax.scatter(FEAT[clave]["Kurt_banda"], FEAT[clave]["R_BPFI_dB"], s=18, alpha=0.7,
               color=COLOR[CH[clave]["rol"]], label=clave,
               edgecolors="none")
ax.axvline(UMBRAL["Kurt_banda"], color="k", ls="--", lw=1)
ax.axhline(UMBRAL["R_BPFI_dB"], color="k", ls="--", lw=1)
ax.set(xlabel=r"$\mathrm{Kurt}_b$ (genérico)", ylabel=r"$R_{BPFI}$ [dB] (específico)",
       title="Espacio de indicadores: cada punto es una ventana de 1 s")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

*Figura 22. Distribución de los tres indicadores por ventana de 1 s y por canal, con el umbral $\mu_{ref} + 3\sigma_{ref}$ a trazos: 1.768 para $\text{Kurt}_b$ y 1.578 dB para $R_{\text{BPFI}}$. El de $\text{SNR}_{\text{banda}}$ (6 dB) es un criterio de validez de la medición, no de falla. En $\text{Kurt}_b$ y $R_{\text{BPFI}}$ la distribución de M2-C1 queda mayormente a la derecha del umbral, y en $\text{SNR}_{\text{banda}}$ el ruido queda solo, cerca de 0 dB.*

*Figura 23. Las mismas ventanas en el plano $\text{Kurt}_b$–$R_{\text{BPFI}}$. El cuadrante superior derecho, donde los dos indicadores superan su umbral, tiene el 85.0 % de las ventanas de M2-C1 y ninguna de los canales de referencia (se imprime abajo).*

In [ ]:
# Separabilidad: cuantas ventanas de cada canal superan cada umbral
filas = []
for clave in CH:
    fila = {"canal": clave}
    for col in FEAT_FALLA:
        fila[f"% > umbral ({col})"] = 100 * np.mean(FEAT[clave][col] > UMBRAL[col])
    filas.append(fila)
display(pd.DataFrame(filas).set_index("canal").style.format("{:.1f}"))

# Indice de Fisher: separacion entre medias relativa a la dispersion conjunta
print("Separabilidad falla vs. referencia (indice de Fisher):")
print("  J = (mu_falla - mu_ref)^2 / (var_falla + var_ref)")
for col in FEAT_FALLA:
    a, b = FEAT["M2-C1"][col], ref[col]
    J = (a.mean() - b.mean()) ** 2 / (a.var() + b.var())
    print(f"  {col:<14}: J = {J:8.2f}")

print("\nVentanas con AMBOS indicadores sobre umbral (el cuadrante de falla):")
for clave in CH:
    conj = np.mean((FEAT[clave]["Kurt_banda"] > UMBRAL["Kurt_banda"]) &
                   (FEAT[clave]["R_BPFI_dB"] > UMBRAL["R_BPFI_dB"]))
    print(f"  {clave:<6}: {100 * conj:5.1f} %")


def energia_abs(x, fs, banda, f_obj, n_arm=3, tol_rel=0.03):
    # el indicador descartado: energia ABSOLUTA de la envolvente en f_obj y armonicos
    sos = signal.butter(6, banda, btype="bandpass", fs=fs, output="sos")
    env = np.abs(signal.hilbert(signal.sosfiltfilt(sos, x)))
    nps = 1 << int(np.floor(np.log2(len(env) / 4)))
    fv, Pv = signal.welch(env - env.mean(), fs, nperseg=nps, noverlap=nps // 2)
    tol = max(tol_rel * f_obj, 3 * (fv[1] - fv[0]))
    e = 0.0
    for k in range(1, n_arm + 1):
        mk = np.abs(fv - k * f_obj) <= tol
        e += np.trapezoid(Pv[mk], fv[mk])
    return e


E_abs = {k: np.array([energia_abs(v, FS_ACEL, BANDA, f_pico)
                      for v in segmentar(CH[k]["x"], FS_ACEL, T_VENTANA)])
         for k in ["M2-C1"] + CANALES_REF}
a, b = E_abs["M2-C1"], np.concatenate([E_abs[k] for k in CANALES_REF])
print(f"\nIndicador descartado (energia ABSOLUTA en f_BPFI): J = "
      f"{(a.mean() - b.mean()) ** 2 / (a.var(ddof=1) + b.var(ddof=1)):.2f}   "
      f"(mediana M2-C1 {np.median(a):.2e}, M2-C2 {np.median(E_abs['M2-C2']):.2e})")

# tolerancia de busqueda: el BPFI teorico con +-3 % no alcanza al pico medido
tol3 = 0.03 * TEO["BPFI"]
print(f"\nBPFI teorico {TEO['BPFI']:.2f} Hz, tolerancia +-3 % = +-{tol3:.2f} Hz; el pico medido "
      f"esta a {TEO['BPFI'] - f_pico:.2f} Hz -> "
      f"{'adentro' if TEO['BPFI'] - f_pico <= tol3 else 'AFUERA'}")

En el diagrama de dispersión, las ventanas del canal con falla ocupan el cuadrante superior derecho (alta impulsividad y alta relación pico/fondo en $f_{\text{BPFI}}$), separadas del grupo de los canales sanos. En estos datos, **la regla conjunta no separa mejor que $R_{\text{BPFI}}$ solo**: los dos tienen cero falsas alarmas sobre la referencia, con 85.0 % y 88.3 % de detección. $\text{Kurt}_b$ solo detecta más (96.7 %), pero con falsas alarmas (5.0 % de las ventanas de M2-C2). Por qué igual los combinamos se explica en el paso 10. El ruido aparece con relación pico/fondo baja, como corresponde, y además queda descartado por el criterio de validez de la medición.

Una aclaración sobre la tolerancia de búsqueda, y sobre qué está sesgado y qué no. La forma de onda de aceleración es cruda: nada en la cadena de medición ni en los pasos 1 a 7 favorece una frecuencia sobre otra. El pico en 173.2 Hz sale de buscar el máximo en una ventana ancha ($2f_r$ a $15f_r$), y la comparación con el reporte SKF se hace después, para interpretarlo. Si el rodamiento fuera otro, el mismo código encontraría otro pico.

Lo que sí introduce una frecuencia de referencia es este paso. $R_{\text{BPFI}}$ es, por construcción, un indicador sintonizado: mide la relación pico/fondo dentro de $f_{\text{obj}} \pm 3\%$, y acá le pasamos $f_{\text{obj}} = 173.2$ Hz, el valor medido en el paso 7. Eso está bien para vigilar una falla ya identificada.

El riesgo aparece al llevar esto a un monitor automático. Un sistema configurado con los datos del rodamiento usaría el BPFI teórico (180.4 Hz) como $f_{\text{obj}}$, y la ventana de $\pm3\%$ (±5.41 Hz) no llegaría a cubrir el pico real, que está 7.17 Hz más abajo por el deslizamiento: devolvería "sano" sobre un rodamiento fallado. La tolerancia tiene que ser de al menos $\pm5\%$ alrededor del valor teórico para absorber el *slip*. Es una limitación del extractor de características, no de la medición.

### Curvas ROC y DET: cómo elegir el umbral

El umbral $\mu_{\text{ref}} + 3\sigma_{\text{ref}}$ es una elección, y **cada elección es un compromiso entre dos errores**. Si un indicador $s$ se compara con un umbral $u$,

$$P_{FA}(u) = P(s > u \mid \text{sano}), \qquad P_{\text{miss}}(u) = P(s \le u \mid \text{falla}), \qquad P_D = 1 - P_{\text{miss}}$$

Bajar el umbral detecta más fallas pero da más falsas alarmas, y subirlo hace lo contrario. En vez de mirar un solo punto, barremos todos los umbrales posibles.

La curva ROC grafica $P_D$ contra $P_{FA}$. El área bajo la curva (AUC) es la probabilidad de que una ventana con falla tenga un puntaje mayor que una sana: 1 es separación perfecta y 0.5 es lo mismo que tirar una moneda. La curva DET muestra lo mismo como $P_{\text{miss}}$ contra $P_{FA}$, con los dos ejes en escala probit ($\Phi^{-1}$, el cuantil de la normal). Así se ven los errores chicos, que en la ROC quedan aplastados contra la esquina; si los puntajes de las dos clases fueran gaussianos, la DET sería una recta. Como resumen de un número usamos la tasa de error igual (EER), el punto donde $P_{FA} = P_{\text{miss}}$.

Las clases son las ventanas de 1 s: 60 de M2-C1 (falla) y 180 de los tres canales de referencia (sanos). El ruido queda afuera porque la regla lo descarta antes como medición no válida. Comparamos $\text{Kurt}_b$, $R_{\text{BPFI}}$, el $C_{\text{BPFI}}$ del cepstrum, la energía absoluta en BPFI que habíamos descartado y la regla conjunta. La regla conjunta también tiene un puntaje: pedir que $\text{Kurt}_b$ y $R_{\text{BPFI}}$ superen los dos $\mu + k\sigma$ es lo mismo que pedir que el menor de sus z-scores supere $k$, así que usamos $\min(z_K, z_R)$.

In [ ]:
from scipy.stats import norm

CLASE_FALLA = "M2-C1"                  # 60 ventanas "positivas"; las negativas son CANALES_REF
# El registro de ruido queda afuera: la regla lo descarta antes como medicion no valida.


def curva_roc(pos, neg):
    # P_FA y P_D para todos los umbrales posibles, de mayor a menor.
    u = np.r_[np.inf, np.unique(np.r_[pos, neg])[::-1]]
    p_fa = np.array([np.mean(neg >= t) for t in u])
    p_d = np.array([np.mean(pos >= t) for t in u])
    return p_fa, p_d


def auc(pos, neg):
    # Area bajo la ROC = P(puntaje de una ventana con falla > puntaje de una sana).
    d = pos[:, None] - neg[None, :]
    return np.mean(d > 0) + 0.5 * np.mean(d == 0)


def eer(p_fa, p_d):
    # Tasa de error donde P_FA = P_miss.
    j = np.argmin(np.abs(p_fa - (1 - p_d)))
    return 0.5 * (p_fa[j] + 1 - p_d[j])


def por_canal(get):
    # (puntajes de la clase falla, puntajes de la referencia sana)
    return (np.asarray(get(CLASE_FALLA), float),
            np.concatenate([np.asarray(get(k), float) for k in CANALES_REF]))


def umbral_k(neg, k):
    return neg.mean() + k * neg.std(ddof=1)


# La regla conjunta "Kurt_b y R_BPFI sobre umbral" es lo mismo que pedir que el MENOR de
# los dos z-scores (respecto de la referencia) supere k. Ese minimo es su puntaje.
def z_and(clave):
    zk = (FEAT[clave]["Kurt_banda"] - ref["Kurt_banda"].mean()) / ref["Kurt_banda"].std()
    zr = (FEAT[clave]["R_BPFI_dB"] - ref["R_BPFI_dB"].mean()) / ref["R_BPFI_dB"].std()
    return np.minimum(zk, zr)


# nombre -> (positivos, negativos, umbral en funcion de k, color)
PUNT = {}
for nombre, get, color in [
        ("Kurt_b", lambda k: FEAT[k]["Kurt_banda"], "#2471a3"),
        ("R_BPFI", lambda k: FEAT[k]["R_BPFI_dB"], COLOR["falla"]),
        ("C_BPFI (cepstrum)", lambda k: CEP[k], "#8e44ad"),
        ("E_abs (descartado)", lambda k: E_abs[k], "#7f8c8d")]:
    pos, neg = por_canal(get)
    PUNT[nombre] = (pos, neg, lambda k, neg=neg: umbral_k(neg, k), color)
pos, neg = por_canal(z_and)
PUNT["Kurt_b y R_BPFI"] = (pos, neg, lambda k: k, "k")

fig, ax = plt.subplots(1, 2, figsize=(13, 5.2))
eps = 0.5 / len(PUNT["R_BPFI"][1])     # medio error: donde se dibuja un 0 % en escala probit
filas = []
for nombre, (pos, neg, umb, color) in PUNT.items():
    p_fa, p_d = curva_roc(pos, neg)
    u3 = umb(K_SIGMA)
    fa3, d3 = np.mean(neg > u3), np.mean(pos > u3)
    ls = "--" if "descartado" in nombre else "-"
    lbl = f"{nombre} (AUC = {auc(pos, neg):.4f})"
    ax[0].plot(p_fa, p_d, ls, color=color, lw=1.5, drawstyle="steps-post", label=lbl)
    ax[0].plot(fa3, d3, "o", color=color, ms=7, mec="w")

    pf = norm.ppf(np.clip(p_fa, eps, 1 - eps))
    pm = norm.ppf(np.clip(1 - p_d, eps, 1 - eps))
    ax[1].plot(pf, pm, ls, color=color, lw=1.5, drawstyle="steps-post")
    ax[1].plot(norm.ppf(np.clip(fa3, eps, 1)), norm.ppf(np.clip(1 - d3, eps, 1)), "o",
               color=color, ms=7, mec="w")
    filas.append({"indicador": nombre, "AUC": auc(pos, neg),
                  "EER [%]": 100 * eer(p_fa, p_d),
                  f"P_D con k={K_SIGMA:.0f} [%]": 100 * d3,
                  f"P_FA con k={K_SIGMA:.0f} [%]": 100 * fa3,
                  "J (Fisher)": (pos.mean() - neg.mean()) ** 2 / (pos.var(ddof=1) + neg.var(ddof=1))})

ax[0].plot([0, 1], [0, 1], color="0.75", lw=0.8, ls=":")
ax[0].set(xlabel="$P_{FA}$ (ventanas sanas sobre el umbral)",
          ylabel="$P_D$ (ventanas con falla sobre el umbral)",
          xlim=(-0.01, 1), ylim=(0, 1.01), title="ROC")
ax[0].legend(loc="lower right", fontsize=8)

ticks = np.array([0.01, 0.02, 0.05, 0.1, 0.2, 0.4, 0.6, 0.8])
lim = (norm.ppf(eps) - 0.15, norm.ppf(0.9))
for set_t, set_l in [(ax[1].set_xticks, ax[1].set_xticklabels),
                     (ax[1].set_yticks, ax[1].set_yticklabels)]:
    set_t(norm.ppf(ticks))
    set_l([f"{100 * t:g}" for t in ticks])
ax[1].axvspan(lim[0], norm.ppf(eps) + 0.05, color="0.92", zorder=0)
ax[1].axhspan(lim[0], norm.ppf(eps) + 0.05, color="0.92", zorder=0)
ax[1].set(xlabel="$P_{FA}$ [%]", ylabel="$P_{miss}$ [%]", xlim=lim, ylim=lim,
          title="DET (escala probit; franja gris = 0 errores)")
fig.suptitle(f"{len(PUNT['R_BPFI'][0])} ventanas con falla contra "
             f"{len(PUNT['R_BPFI'][1])} sanas. Circulo: umbral actual "
             f"$\\mu_{{ref}} + {K_SIGMA:.0f}\\sigma_{{ref}}$", y=1.0, fontsize=11)
fig.tight_layout()
plt.show()

display(pd.DataFrame(filas).set_index("indicador").style.format("{:.4f}", subset=["AUC"])
        .format("{:.1f}", subset=[c for c in filas[0] if c not in ("indicador", "AUC", "J (Fisher)")])
        .format("{:.2f}", subset=["J (Fisher)"]))

*Figura 24. ROC (izquierda) y DET (derecha) de los cinco puntajes, con 60 ventanas con falla y 180 sanas. El círculo marca el umbral actual, $\mu_{ref} + 3\sigma_{ref}$. En la DET, la franja gris es donde se dibujan los errores de 0 %, que en escala probit no tienen lugar: con 180 ventanas sanas, la menor tasa distinta de cero es 0.56 %.*

In [ ]:
# Eleccion del umbral: como cambian los dos errores al mover k en mu_ref + k*sigma_ref.
ks = np.linspace(0, 6, 241)
fig, ax = plt.subplots(1, 2, figsize=(12.5, 3.8), sharex=True, sharey=True)
for nombre, (pos, neg, umb, color) in PUNT.items():
    if "descartado" in nombre:
        continue
    p_fa = np.array([np.mean(neg > umb(k)) for k in ks])
    p_miss = np.array([np.mean(pos <= umb(k)) for k in ks])
    ax[0].plot(ks, 100 * p_fa, color=color, lw=1.5, label=nombre)
    ax[1].plot(ks, 100 * p_miss, color=color, lw=1.5, label=nombre)
for a, tit in zip(ax, ["Falsas alarmas: $P_{FA}$", "Fallas no detectadas: $P_{miss}$"]):
    a.axvline(K_SIGMA, color="k", ls="--", lw=1)
    a.set(xlabel="k  (umbral = $\\mu_{ref} + k\\,\\sigma_{ref}$)", title=tit)
ax[0].set_ylabel("% de ventanas")
ax[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

print("Umbral mas bajo sin falsas alarmas sobre la referencia, y lo que se pierde ahi:")
for nombre, (pos, neg, umb, color) in PUNT.items():
    if "descartado" in nombre:
        continue
    sin_fa = np.array([np.mean(neg > umb(k)) == 0 for k in ks])
    con_k = f"(con k = {K_SIGMA:.0f}: P_miss = {100 * np.mean(pos <= umb(K_SIGMA)):5.1f} %)"
    if not sin_fa.any():
        print(f"  {nombre:<18}: no llega a 0 falsas alarmas con k <= {ks[-1]:.0f}   {con_k}")
        continue
    k0 = ks[np.argmax(sin_fa)]
    print(f"  {nombre:<18}: k = {k0:.2f} -> P_miss = {100 * np.mean(pos <= umb(k0)):5.1f} %   "
          + con_k)

# Validacion dejando un canal afuera: el umbral se calcula con dos canales de referencia
# y la falsa alarma se mide en el tercero, que no participo en fijarlo.
GET = {"Kurt_b": lambda k: FEAT[k]["Kurt_banda"].to_numpy(),
       "R_BPFI": lambda k: FEAT[k]["R_BPFI_dB"].to_numpy(),
       "C_BPFI": lambda k: CEP[k]}
filas = []
for fuera in CANALES_REF:
    resto = [k for k in CANALES_REF if k != fuera]
    fila = {"canal evaluado": fuera}
    z_pos, z_out = [], []
    for nombre, get in GET.items():
        r = np.concatenate([get(k) for k in resto])
        u = umbral_k(r, K_SIGMA)
        fila[f"P_FA {nombre}"] = 100 * np.mean(get(fuera) > u)
        fila[f"P_D {nombre}"] = 100 * np.mean(get(CLASE_FALLA) > u)
        if nombre != "C_BPFI":
            z_pos.append((get(CLASE_FALLA) - r.mean()) / r.std(ddof=1))
            z_out.append((get(fuera) - r.mean()) / r.std(ddof=1))
    fila["P_FA ambos"] = 100 * np.mean(np.minimum(*z_out) > K_SIGMA)
    fila["P_D ambos"] = 100 * np.mean(np.minimum(*z_pos) > K_SIGMA)
    filas.append(fila)
print(f"Dejando un canal de referencia afuera (k = {K_SIGMA:.0f}), en % de ventanas:")
display(pd.DataFrame(filas).set_index("canal evaluado").style.format("{:.1f}"))

*Figura 25. Falsas alarmas (izquierda) y fallas no detectadas (derecha) en función de $k$, con el umbral $\mu_{ref} + k\sigma_{ref}$ (para la regla conjunta, $\min(z_K, z_R) > k$). La línea a trazos es el $k = 3$ que usamos.*

Con todos los umbrales a la vista, el orden de los indicadores queda claro. $R_{\text{BPFI}}$ y la regla conjunta son casi perfectos (AUC de 0.9995 y 0.9999, EER de 1.7 % y 0.3 %), y $\text{Kurt}_b$ queda apenas atrás (AUC 0.9923, EER 3.1 %). La energía absoluta da un AUC de 0.317, peor que tirar una moneda: ordena al revés, porque el canal sano M2-C2 tiene más energía en esa frecuencia que el fallado.

El cepstrum queda por debajo de $R_{\text{BPFI}}$ en todas las medidas: AUC 0.977, EER 5.3 % y $J$ = 3.42 contra 5.25. Con $k = 3$ detecta solo el 28.3 % de las ventanas, porque la ventana de M2-C2 con valor 9.02 infla el $\sigma_{\text{ref}}$ y sube el umbral. Probamos otras resoluciones espectrales y otros entornos para estimar el fondo, y $J$ quedó siempre entre 0.9 y 3.4. En este registro, entonces, **el cepstrum confirma la periodicidad pero no detecta mejor que la relación pico/fondo del espectro de envolvente**. Tendría más sentido con un defecto más avanzado, que genera muchos armónicos y bandas laterales, o para buscar periodicidades sin conocer la frecuencia de antemano.

Sobre el umbral, la Figura 25 muestra que $k = 3$ es conservador. $R_{\text{BPFI}}$ deja de tener falsas alarmas sobre la referencia desde $k = 2.10$, y ahí pierde el 3.3 % de las ventanas con falla en vez del 11.7 %. La regla conjunta llega a cero falsas alarmas desde $k = 1.95$, también con 3.3 % de fallas perdidas contra 15.0 % con $k = 3$. $\text{Kurt}_b$, en cambio, no llega a cero falsas alarmas con ningún $k$ hasta 6, por las ventanas impulsivas sueltas de M2-C2. Igual no cambiamos $k$: bajarlo sería ajustar el umbral a estos mismos datos, y con 180 ventanas de tres canales la diferencia entre 2 y 3 no está bien estimada.

Queda una objeción: las falsas alarmas se midieron sobre los mismos datos que fijaron el umbral. Para eso está la validación dejando un canal afuera, que calcula el umbral con dos canales de referencia y evalúa en el tercero. $R_{\text{BPFI}}$ y la regla conjunta siguen sin falsas alarmas en los tres casos, con detección de 88.3 a 90.0 % y de 76.7 a 85.0 %. $\text{Kurt}_b$ mantiene un 5.0 % de falsas alarmas en M2-C2, el canal del mismo equipo. Es la estimación más honesta que permiten los datos, pero sigue siendo un solo rodamiento: las curvas dicen qué tan bien se separan las ventanas de esta medición, no cómo funcionaría el método en otro equipo.

---

# 10. Resumen y veredicto automático

Por último juntamos todo en una tabla comparativa con una regla de decisión. El veredicto combina el indicador genérico (kurtosis: hay algo impulsivo) con el específico (relación pico/fondo en $f_{\text{BPFI}}$: ese algo es esta falla):

| Condición | Veredicto |
|---|---|
| $\text{Kurt}_b$ y $R_{\text{BPFI}}$ sobre umbral en la mayoría de las ventanas | FALLA |
| uno de los dos sobre umbral, o ambos de forma intermitente | ALERTA |
| ninguno | OK |
| $\text{SNR}_{\text{banda}}$ por debajo del mínimo | MEDICIÓN NO VÁLIDA |

**Exigir las dos condiciones a la vez reduce las falsas alarmas**: un golpe accidental sube la kurtosis pero no deja energía en $f_{\text{BPFI}}$ con armónicos, y una resonancia estacionaria puede dar un pico de envolvente sin impulsividad asociada.

In [ ]:
def veredicto(feat, umbral, frac_min=0.5):
    # Regla de decision sobre la poblacion de ventanas de un canal.
    if feat["SNR_banda_dB"].median() < umbral["SNR_banda_dB"]:
        return "MEDICION NO VALIDA"
    f_k = np.mean(feat["Kurt_banda"] > umbral["Kurt_banda"])
    f_r = np.mean(feat["R_BPFI_dB"] > umbral["R_BPFI_dB"])
    if f_k >= frac_min and f_r >= frac_min:
        return "FALLA"
    if f_k >= frac_min or f_r >= frac_min:
        return "ALERTA"
    return "OK"


filas = []
for clave, d in CH.items():
    ind = indicadores_tiempo(d["x"])
    fe, Pe = ENVC[clave]["f"], ENVC[clave]["P"]          # banda comun, comparable
    _, _, snr_pico = pico_cerca(fe, Pe, f_pico, max(0.03 * f_pico, 3 * (fe[1] - fe[0])))
    ft = FEAT[clave]
    filas.append({
        "canal": clave,
        "descripcion": d["desc"],
        "RMS [g]": ind["RMS"],
        "Cresta": ind["Cresta"],
        "Kurt cruda": ind["Kurtosis"],
        "Kurt banda": ft["Kurt_banda"].median(),
        "banda kurtograma [Hz]": f"{KURT[clave]['banda'][0]:.0f}-{KURT[clave]['banda'][1]:.0f}",
        "SK max": KURT[clave]["SK"],
        f"pico env @ {f_pico:.0f} Hz [dB]": snr_pico,
        "R_BPFI [dB]": ft["R_BPFI_dB"].median(),
        "SNR banda [dB]": ft["SNR_banda_dB"].median(),
        "VEREDICTO": veredicto(ft, UMBRAL),
    })

resumen = pd.DataFrame(filas).set_index("canal")


def colorear(v):
    return {"FALLA": "background-color:#f5b7b1; font-weight:bold",
            "ALERTA": "background-color:#fdebd0; font-weight:bold",
            "OK": "background-color:#d5f5e3",
            "MEDICION NO VALIDA": "background-color:#e5e7e9"}.get(v, "")


display(resumen.style
        .format({"RMS [g]": "{:.4f}", "Cresta": "{:.2f}", "Kurt cruda": "{:+.3f}",
                 "Kurt banda": "{:+.3f}", "SK max": "{:.2f}",
                 f"pico env @ {f_pico:.0f} Hz [dB]": "{:.1f}",
                 "R_BPFI [dB]": "{:+.2f}", "SNR banda [dB]": "{:.1f}"})
        .map(colorear, subset=["VEREDICTO"]))

In [ ]:
print("=" * 74)
print("  INFORME DE DIAGNOSTICO")
print("=" * 74)
print(f"  Equipo         : ventilador industrial, rodamiento SKF 22230 CCK/W33")
print(f"  Velocidad      : {fr:.3f} Hz ({fr*60:.0f} rpm), medida del espectro de velocidad")
print(f"  Registro       : {T_ANALISIS:.0f} s a {FS_ACEL/1000:.0f} kHz")
print("-" * 74)
print(f"  Banda de demodulacion (kurtograma) : {BANDA[0]:.0f} - {BANDA[1]:.0f} Hz "
      f"(SK = {KURT['M2-C1']['SK']:.2f})")
print(f"  Pico dominante de envolvente       : {f_pico:.2f} Hz = {f_pico/fr:.3f} x fr")
print(f"  Armonicos detectados               : {int((ARM['SNR_dB'] > 3).sum())} de {len(ARM)}")
print(f"  Bandas laterales a +-fr            : {int((LAT['SNR_dB'] > 3).sum())} de {len(LAT)}")
print(f"  Candidato teorico mas proximo      : {mejor_match} "
      f"({COEF[mejor_match]:.3f} x fr, error {err:+.2f} %)")
print("-" * 74)
for clave in CH:
    print(f"  {clave:<6} {resumen.loc[clave, 'VEREDICTO']:<20} "
          f"kurt_b={resumen.loc[clave, 'Kurt banda']:+6.3f}  "
          f"R_BPFI={resumen.loc[clave, 'R_BPFI [dB]']:+6.2f} dB  "
          f"RMS={resumen.loc[clave, 'RMS [g]']:.4f} g")
print("-" * 74)
print(f"  DIAGNOSTICO: falla localizada en PISTA INTERNA (BPFI) del rodamiento")
print(f"               del lado acople del ventilador 2 (canal M2-C1).")
print("=" * 74)

---

# Conclusiones

El canal M2-C1 tiene una **falla localizada en la pista interna** del rodamiento. La evidencia es:

1. kurtosis de 1.098 frente a −0.006, −0.015 y 0.577 en los canales de referencia, positiva en todas las ventanas del registro;
2. un pico dominante en el espectro de envolvente a $10.4\times f_r$, múltiplo no entero de la velocidad de giro, lo que descarta las fuentes sincrónicas;
3. cuatro armónicos de ese pico, propios de un tren de impactos periódico;
4. bandas laterales a $\pm f_r$ (7 de 8 sobre 3 dB), que indican una fuente que gira con el eje, es decir pista interna y no externa;
5. coincidencia con el BPFI del reporte SKF dentro de un $4\%$, atribuible al deslizamiento y al ángulo de contacto real;
6. el mismo pico en la envolvente que calcula por su cuenta el software de mantenimiento de la planta.

Desde el punto de vista del procesamiento de señales, el trabajo deja varias cosas.

El nivel global de vibración, que es el indicador de la norma, señala el canal equivocado: acá el canal sano tiene el doble de RMS que el fallado. Para detectar la falla hay que mirar la forma de la señal (kurtosis) y no su energía.

El espectro convencional no resuelve el problema. La energía de los impactos se reparte en la banda de resonancia y la línea en BPFI queda enterrada. Ni Welch ni un modelo AR la recuperan, y no por falta de resolución, sino porque la falla es una modulación y hay que demodular. El AR solo aporta cuando el registro es corto.

La kurtosis espectral convierte la elección de la banda de demodulación, que es lo más subjetivo del método, en una optimización con un criterio explícito. Pero necesita guardas, porque mide contraste y no energía: el registro de ruido llega a $SK = 243.86$, 124 veces el del canal con falla. Además, la banda que elige no es la del *hump* visible, que también está en los canales sanos. Un máximo de SK, solo, no es un diagnóstico.

El diagnóstico no sale de una fórmula con geometría supuesta sino de la estructura del espectro de envolvente: múltiplo no entero, armónicos y bandas laterales a $\pm f_r$. El reporte del fabricante confirma, pero no determina.

Un indicador espectral de falla tiene que estar normalizado por el fondo local. La energía absoluta en $f_{\text{BPFI}}$ no separó nada ($J = 0.22$) porque mide el nivel de vibración del canal y no la falla. La relación pico/fondo en la misma frecuencia da $J = 5.25$ y 88.3 % de detección sin falsas alarmas sobre la referencia. Es el mismo error que usar el RMS, pero un nivel más adentro del procesamiento.

La tolerancia con que se busca la frecuencia de falla tiene que absorber el deslizamiento: buscando el BPFI teórico con $\pm3\%$, el pico real habría quedado afuera.

Las curvas ROC y DET muestran que la detección no depende de un umbral afinado: $R_{\text{BPFI}}$ tiene un AUC de 0.9995 y sigue sin falsas alarmas cuando el umbral se fija sin el canal que se evalúa. El cepstrum de la envolvente confirma la periodicidad de 5.77 ms, pero separa peor que la relación pico/fondo ($J$ = 3.42 contra 5.25): en este registro hay pocos armónicos, y el logaritmo le da al ruido el mismo peso que a ellos.

### Limitaciones

Hay una sola medición por estado y no hay histórico. La tasa de falsas alarmas solo la pudimos estimar dejando un canal de referencia afuera (paso 9), con tres canales y un único rodamiento fallado, y no podemos seguir la tendencia de la falla; el umbral $\mu + 3\sigma$ es un prototipo metodológico. Los canales "sanos" son un punto del mismo equipo y dos de otro ventilador, no el mismo punto en estado sano: la comparación es razonable, pero no es un control ideal. $Z=19$ se dedujo de las identidades de consistencia sobre el reporte SKF, porque no tenemos la geometría interna de fábrica; la celda de revalidación del paso 7 queda lista para cuando se consiga. Y no estimamos la severidad del defecto ni la vida remanente, que requieren seguimiento en el tiempo y no una medición aislada.

### Cómo seguir

Lo natural es repetir la medición periódicamente y seguir la evolución de $R_{\text{BPFI}}$. Que crezcan los armónicos de orden alto y que suba el piso de ruido del espectro de envolvente son los indicadores clásicos de que el defecto avanza hacia uno distribuido.